In [ ]:
import base64, csv, hashlib, json, os, platform, re, shutil, signal, subprocess, sys
import threading, time, urllib.error, urllib.request, zipfile, zlib
from datetime import datetime, timezone
from pathlib import Path

if not sys.platform.startswith("linux") or not Path("/kaggle/working").is_dir():
    raise RuntimeError("本 Notebook 应在 Kaggle 的 Linux 会话里运行。")

NOTEBOOK_START = time.time()

# ---------- 可以改的设置 ----------
OLLAMA_MODELS = ["qwen3.5:0.8b", "qwen3.5:2b", "qwen3.5:4b", "qwen3.5:9b"]
N_SUBSET = 200               # 慢的组只跑验证集前 N_SUBSET 条
OLLAMA_BUDGET_HOURS = 9.0    # 从开始运行算起，超过后不再开始新的 Ollama 实验
MIN_START_MINUTES = 2        # 离上面的时间点不到这么多分钟时，不再开始新的一组
DEBERTA_BUDGET_HOURS = 10.0  # 超过后 DeBERTa 不再开始新的一组
FINAL_WAIT_HOURS = 10.8      # 最晚等 DeBERTa 到这个时间，然后汇总打包（Kaggle 一次最长 12 小时）
RUN_OLLAMA = True
RUN_DEBERTA = True
DEBERTA_EXTRA_ARGS = []      # 例如 ["--sizes", "16,64,full"]；默认 16/64/256/1024/全量 × 3 个种子
DEBERTA_TRAIN_FILE = None    # None 表示从 Hugging Face 下载 SST-2 训练集
DEBERTA_MODELS = ["microsoft/deberta-v3-base", "microsoft/deberta-base"]  # 第一个冒烟测试不通过时改用第二个
DEBERTA_PIP = ["transformers==5.19.0", "datasets==5.1.0", "sentencepiece==0.2.2", "protobuf>=4.25,<8"]
OLLAMA_VERSION = "0.40.3"

# ---------- 路径 ----------
ROOT = Path("/kaggle/working/sst2_remaining")
DATA_DIR, META_DIR, SCRIPT_DIR = ROOT / "data", ROOT / "metadata", ROOT / "scripts"
OLLAMA_RESULTS, DEBERTA_DIR = ROOT / "ollama_results", ROOT / "deberta"
MODEL_DIR = Path("/tmp/sst2_ollama_models")  # 模型权重放在这里，不进结果包
for directory in (ROOT, DATA_DIR, META_DIR, SCRIPT_DIR, OLLAMA_RESULTS, DEBERTA_DIR, MODEL_DIR):
    directory.mkdir(parents=True, exist_ok=True)
EVAL_SCRIPT = SCRIPT_DIR / "sst2_instruction_eval.py"
DEBERTA_SCRIPT = SCRIPT_DIR / "deberta_sst2.py"
DATA = DATA_DIR / "sst2_validation.tsv"
HOST = "http://127.0.0.1:11434"
ZIP_PATH = Path("/kaggle/working/sst2_remaining_results.zip")


# ---------- Ollama 实验顺序 ----------
def experiment(group, model, prompt, think, n, cap_minutes):
    return {"group": group, "model": model, "prompt": prompt, "think": think,
            "n": n, "cap_minutes": cap_minutes}


OTHERS = [m for m in OLLAMA_MODELS if m != "qwen3.5:4b"]
PLAN = ([experiment("P1", m, "basic", "off", 0, 30) for m in OLLAMA_MODELS]
        + [experiment("P2", "qwen3.5:4b", p, t, N_SUBSET, 45 if t == "off" else 60)
           for p, t in (("basic", "on"), ("cot", "off"), ("cot", "on"),
                        ("role_cot", "off"), ("role_cot", "on"))]
        + [experiment("P3", m, "cot", "on", N_SUBSET, 60) for m in OTHERS]
        + [experiment("P4", m, "basic", "on", N_SUBSET, 60) for m in OTHERS]
        + [experiment("P5", m, p, "off", N_SUBSET, 45) for m in OTHERS for p in ("cot", "role_cot")]
        + [experiment("P6", m, "role_cot", "on", N_SUBSET, 60) for m in OTHERS])
PLAN = [e for e in PLAN if e["model"] in OLLAMA_MODELS]


def hours_used():
    return (time.time() - NOTEBOOK_START) / 3600


def save_json(path, value):
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2, default=str), encoding="utf-8")


def tail(path, chars=3000):
    path = Path(path)
    return path.read_text(encoding="utf-8", errors="replace")[-chars:] if path.exists() else ""


STATUS = {"started_at_utc": datetime.now(timezone.utc).isoformat(), "notes": []}


def note(message):
    """打印一条重要信息，同时记进 metadata/notebook_status.json。"""
    print(message, flush=True)
    STATUS["notes"].append({"hours": round(hours_used(), 3), "message": message})
    save_json(META_DIR / "notebook_status.json", STATUS)


print(f"Ollama 实验 {len(PLAN)} 组；DeBERTa：{'运行' if RUN_DEBERTA else '跳过'}")
for i, e in enumerate(PLAN, 1):
    rows = "全部 872 条" if e["n"] == 0 else f"前 {e['n']} 条"
    print(f"  {i:>2}. {e['group']}  {e['model']:<13} {e['prompt']:<9} 思考 {e['think']:<3}  "
          f"{rows}，上限 {e['cap_minutes']} 分钟")

In [ ]:
EMBEDDED_ASSETS = {
    "sst2_instruction_eval.py": {"sha256": "187d2f88d294e70ab427c694dc0dff6f54b85e3eac3849cba43eef15fc0d5906", "bytes": 41027,
        "zlib_base64": "eNrVfft3E9e18O/+K06H22UJJNkG0qZaVbrSQG/4mgcXaHv7GV1d2RphFVlSNRKGCq1lQgy2sbGT8MaER3iVxA9CwssY/y+3Hkn+qf/C3Y9zZs6MRsJJ22+tj9sba2bO2ee1X2fvffbZ9pO+qlXuG8oV+szCMVE6URkpFnb1GIZx8OCh6E7RmDm7sXrXXry/8eJWa/nTxvfn/v7q2sbz2dbSYnP1dOPhbftLeDNjzz3avHFr4/l4Y+l7rtJa+qt99wEXsCfvNi4t2muPEcLpicanN6FKc/Vma+m2ffZM8/ZS8/zZv41/0tNjL7+wX16wl1/ZzycbN+Ybl67+bXx6ILqxtt688NAB11q/3ro9o/fsb+PYicbC1MbzKfvup/b6ROv+qdbK6cbF76C7PUIMxARA2Hw001o+tXn9jDiWzucy6UquWPj7q8m3fw7jvHGbITRunW0uvv77qyno0L9/8Lu9onntU1ExrYqAevyxcWV589p86/IcfILq9sRDeCOiA1AJYEBr8I97v3l2RlilfK6SMBDEYHxnf9IQMFmt02vNhXOt11/YZ1/iXCxdhjmF357pEGJnTHCTor9vQDRm7zQm5x3I2LHoQB80O9kvEqL13c3NG3dE9B3qycwAvGosfiVfDfCAAOSumGh9v9I6NW3f/Wvj/D177qvG5aeNha/tldfYhelb4peVkVzh6DtQXC+18epa6+wje/qhfQ/6Oi1XAt48ewwdgYVrLd+Al40b6+4cjN9ojZ/eeH7enr8Psy+iUYIs7Ffj9sSTxvkH9uTTv7+67hmQbFzAjDavfwdNN1cX7TNX7fM37QfnGpfONha+gf5ii2euNp5cJKyBRW8+mLXnzze/XublzphmyTLNo0LsgV8H8Ze9dKVx6YV4d/8+6B/MgPi4ZBbe3Sd4dDCHmwvjgDSilCuJXMGqpPN5UYQi6RxMHYAs5vPp0TQNC7qw8FJ8TC8YmL10Grt0/mbzwk0GKPrSpVzf8Ei6QtSxvLF6Tw6bJgXRZu4RTAosL6y5/fIL2Qq1iK3YE083Vi+pXtoTr+ylF40FWCDA02vUdGy4VCLspKeoZZaPmeW/jZ/64ENxsFLN5Irw+9gH8NRcnALSev4caeizNSAaaP0PuUKmOGaJ/cUxs3xwxMznCXtx8v4NGEF8z969+w/u3fvbFExY6rd7/wjYZFhHo7FYzBAB/7YhUiP1rn1hT802H11GvAJiujgJAJmpCMuq7Ezh1Jarw0h5KROoMFY6AXiRMUeL7QA/czBj85Ov7Ll7G6/PwX+RXBbO2UszQNyt9Qv29S+31ERB7Oxva2JqFt8C7W8BRIdhTzzcPP1QSBaypZ4wEUAZp1PbRGPqC0APxg0krVdzWwJVKheP5TJmWWFnNDpazJh58ecxs7Ar9lZ895Bs5J8CrT/29pDzsHNIa2VL4MtmqViueKevMT0tyqZVzVcswRygufpF48uFxuOzwOpAntivbrVuP+zpaSzPNVc/tZe+BB5uLzx0KgFL2Fh9at99sfH8HBIfVgIp8IjfM0L/0p6ftRcvvxP7k1Us5J3WASQsWePWMyBopF4No5A857/m5Witn20+PMe83Z6fQW55ZRnEWfOTFwAe39x6htWJsjbPzraefQa8EjnXjXH4rbc/bB3TRs/twxhBmAGk5sJNGB6ze6DczTsobOEHtAb8E1g5CAZ4bD34qvElkMULwBF8HL8MPYIfleJRsyAaF1fgN4yguXhBazllVUdH0+UTNAPAW1wpc6px+RZQKUsirt149qzx7Ezz5l83r3zvAh5fha+oF/Rky8VRkUplq5Vq2UylRG6U1jVdKBQrJFKtnh71rnyklC5bpnqGCVA/R9LWSD43pB6xY+p30VK/yk5Nqzh81Kw4TyecIpWRspnO5ApHnBe5UadWtZyHNmJmuVws+955+iXflc0/V0FM8wiHgQpMwmFLDfG9YrVQMcvqe2G4Wi6bhUqMp8Ip9pt9Bw4eSr338Yf7P9h7aO+eiDhEfdxfLOb3HjeHq5ViOSLG0jnZUCldwZlQtffDY0/PNpEI/oeqDKs+StpOts49BnQH6QP/bU48aEx/yowTcLYTlJ739767Z+8B4Ooh49dmvjgmchasoNAoF2Y2XQFJag2Xc0MwuLSopK2jEehurmxmxFiuMsI1StUKl5X8wxJGj4vlRrZarowAU4H5qpjHKzHxh3KuYgI4mLESTK7JldMlqF4q59IVM38CCo+W8iboTPDRFHJdYka459D7+z76bergob37Zd+zxTIAK1ggxgAJIoK563C6bGareYCUHipS/0xBMHBg6UJGDMOKUC+silmKDp2I4l9Px6nzIMBzBVHMAoBi9cgIMJxKUZgFq4qNinzxSG44nSeA6WHABgSphkW9fffgb6mf7xbS+RN/MakfR3LHgKBwLgStP0wisKVcAasey5ljBC4Dgy+P4kusYgGWAVoXYI6L+TTM34mY1lfjgAn4V8DBwBTkTVGojg4hG88KM0dzHx0goAMRXjZ4HDKhqCiYR4Bij5n8Vb7EBjXgpaKVozJOJ3BkPfsPAH4fOgijq/VIXk5I1/zmG7FTbN7+HuR3c/WqeA8W99/3H0JFpTk1CawWmSYowPdPAedDRn1xBTgU8FbcVNxH9rZ5dpogGkNpKzdsxEVIIusOYRwuHC5s27ZN7HPxNH640LZs/9Lp/pGTrqbXmU/8GA2edNmEszzu1B8ufFQUiqTM46V8ukA8VxTL4k9VwO8sYCS9AIJGukFijeG0tUHHefwPSRMwiTVsxCwMm3XxB1izXqvDVDA1mDSjv+oI+ICkgrgRjrThx1uEH68fNc8vS6GvMAMwoPH4E3tiEreSi/fwfySEGSGGixU/OhjwR+MJnVEEyyE17vD0VisNbMwzC+pL15HsxpF4UPsM7qFQcZm713rweWvq29bSa3vpGg+gXMybqX9kFD1tKqjxx2IV5CyiE+KDifIzi8wWRD5x3VFavTRShJWzAJGDZ+JHTUe9p2fP3g8/Th3c+9GhvR+9t5d4HXBgS2Rz+VHRiyiYZ51gBPZVwDvTw5UqcEyAaUkkJSxLj8IALDGGkmB4pJyzKqNpK1opm+1UQTIlD+oAkk2OCLBUToPISDM4pw2rAGItDt9LRSCOaFTksiIDPHoIiBWfRnOsrgjFz36/DxbE5WiG2kXCatWQGZkp0BPgwRipVEpWvA+3eDFVKAYzbUSEQSozFlIfotk8qDpGJGAYY8XyUbNsQem3oSbSGcgqeBp4u7/O5Q1WxQN7AB2AaUjnR4pWJT4wsHvXbk/7rnbe3rbW8oCn5Z/1uy3TVvTNLb/d/3Z/37EBve2PigXzh7ZZ76bz7IyJjdXzrNYCx7DnLoG2isyzMXkJGGsnJSf18Ucf/DH10e8+hDUtm7hCpVzeDJWN/xrcEU3+aiD0q/jhWP+O8K/+DTAg9euP/3PvHn/Jw4eHisfNzOHaYWt7iGqF3Wrw7nAdq7770cE/7D2Q+vDdA79lpapsQKFsDojusLUDfrKCcjKfHjLzJ0vAknNE0uGT8oPDZhG2tUPx2vCvTgagvygbqoAGSwKHOQXSPcn7ozB2bu9/7v9g33v7DrVPRA8DOzyEPME3iB34Acb8q3jOOhk/mQg7E+COH98Nxn7yt/FPkjgX/wY4AND3QZt/+PjAHpp9/3wq6XdSybaTbKQ62Xyw3Pjy1Em2Yp1sPJ2EJxeeMwYE/A8O4gd1IWwo+B2Guk2QBnMfatlzz3DjdnG9eXOxeeu7xvQVfmNPPARR0PzmPjw2V2+iaJucb1xd2hy/RjY8YX8+IwZ6UnvePfj+XuQ/wPJjo2nY75QBPUI143B1586BnUh9USQzeOwf2OV93K09ZrP9Gc/jwAA+Dhh14HM9GTPLNskUqcohlONxbDKMhsJKFfTuQXiK4KtknHmBYeBGXdrmYrHYL/ukmY7FN8toGEVjYby5Osl0ak+esV+Nw/aksTCFZch2BWCkkePZM9jGw8TDngWAqG03Gilxj4mtksqW4D/AwOVbYOKGat5A/k/9d4iEPtBGgClLAoiVacght2pEDISdWmXW5VRl2AiCYjVsQnGntGGEYzAjuVIorGCrZ7df/m6RssDj/fY2mjGXZuyVx607XzcmHzUuLW68PING529vwxQ5e3b+N0Qbm4jTJTUONYytjMLpLgPzdFeWNQACw5XfGD9oc5wihhKIHznQKE4ym/eiCa+jCLH1IiJ0a0UY1pbfbzx/SZVF6/bD5t2XDl9vTJENenJF2rVnGjNT+OXC1dbSbTL1YjPmcRDwcsgS1eYeke3f3U9QwWyubFVSpOoLUNbY0tJeluw654HkG1eWte0H8X3ZjJQCtehAXbTWr6K1mczPVBDEeyUltX/QXBvnHzbnzwDEjbUF6Bt5Fb5GfwU1BgKrceU8Wl9AaVy80Lg92bh1trW0QpBAQqoWhT17sXkO7VLsMGktzyIkrbarmvdJVJeQCBTMI0jMjdXVxqdz3CM0ESxe3ni+2JxZUE6O6zgbc/Mw45vjU5u3XyCzmr8P6io0B23Z8183n0zDIrVev4al0E1FckHkstNfh+D83CU8OJCMES8DXdAMST7nkE2hWJF1435UZgwzzNFS5YShrUpC5EFFDLHIjoGczeRg/xZiKGGGPAqlqPRgdCCJ7XBVM2+ZBFc1P0obMbcTg6Mxs5AJheNJSRSGOFw5XD5cIO6//b8PG71GuK2nIVBGEFjsSLlYLYUGwqCUIAfm9gaACA1qn0eBOGlB/wbzBZfXbN8u2Yzz5r89fAf7sX3bO9S+AIIW+QLpv5IX4aQT3FA46W9EK85voav5QtKlEppR+DDYnwRWKZuLITMOU1mqRENx2bCjXYGsqgyPhAiQNjPbxOb9S4D4TAdAEPbcJGNv48ILkID23H1JhhIbyQRweq25+qC5ugiSgqvba18ABJYsTDEK8aj3ODCkcDW0wYF4Mu7RmSrpXF6Ozz82TzncTeeGczgXHp0pZpnpMowP4XhrwJYjg25EE7kztuKZFbQ/8cxQTZ4+1YiLJ1DHadmLnOoflHBbgi0O4iqxTzJkqS+AJelyxUJbQwgQLyx+kuC19X+It2mUXmpLjw7ljlSLVcurehJzgqlxVDv/AAMnM6WqeTS4zlMKY6UaxbIXRHunj6XzVZx4/Cpnsz/s1PdMdkorMxBug+SYWRIMNJZHp1gojDgVcowwSIusGuIvqaC2A0OGpuBtfQW2tApeVhMI2WU3IYPkpEHkaxZCRBzElQYkIbviUQ4DBZnDCIBpbo0TvIlsECrD18mF2upALlTjh5NLJzLpxKs7U47LszXZbvS8iQrckW4N+93yb8b6fxzb1eBp7AF4rrZFhN20IcJfvCdScxKlSUH4Rk87yuYKFFZh9HTbymP8wTL6khqzSx037qiD5ovpTKpcHLNC6fIRS1M+UfQP4qY3Gax8DtaMXOY4dl4ZrwxaSVBmjRAqluF6RMAmJon7Lu5I48a9xsv51vJ3IJ+0HQhMFLYdAyRJp7Kw2XTXA502sBzorAl5y7gznjHzaGQincg4zKSI9WIW7Mlyx73TH6tYx7CfscrxippuI6IRf3oMZXnSeUFGZTTShBBmRMBAi+gHSxjVSjb6dtTKHQF4BXMMSTYBagQa2LJe1oNuKeresHUstgem9AC9CGUjbucTzq82To0UVguY5XosZ1nVIcushLgFUNHMfKaQHgXeATg+mAxggdYJK2Yeh81N1qjhiOrCXp/YvL1qz0yAHipUK7RZpmZAx71rT1728WBUCHKwOcdZNQtoA0Wdk7sR0CpAIntN7Ah0VnY/zNvNsGfDFDB02FK5xF82yDhyOLMD5gHgEBDYJ4XoNxbPEeeJiH7ctAWLgPY5QCNzTU4iLiQyI9iD3KZgGZ4GniZ0Sl/8DtCZQ4UmL6k9FFAb7qhqUPgn5XqAxHJmNiHkNDgrGjh0pzzwWFJDYKA4PvVeTds/ZYhOY6wbws6x+deXAf0C8oilSyXU3EOw/KpaxFmCcNjLS1EzxjkhhElF8H/4BIsEoOo92nj78aWsI302zgvvGNvH1nj2rHnhIezzaVUQdXFVhKPxtpalRUiGuE08hQ0aPMIOs3HrFWyataH+xSwXUzjfCU+XQM+mFafif381SUFrHC8lgKf8/dUUOlUodEyixHWMYLpxz155zHFrUBummqK/Hrr8BqODEoqZxpGmXLyIC8uxLgTYtZmM4krQEo1hp6VcxXfOYOgdlKi3wZGUbOkL47I/q1gtE85CP4gDSskL4OL66hFzHsmmRnMYBOBdryIsV+FYrlwsgEiugNBJV/OA/u//JrX3oz37P9730SHkacraP5KNMhAy9bvrUimf8IIlNyPKAwDqBAWQOJMve1wtedgsVcQ+KrK3vYMOQhnNVy/tlc9cqBQxdk0PVlOf0OI2eak1PsEhd/g6inIJowZgZw9rby+sSMzoNooMLr/ea1CFT1Sjo/l83xFAPJKvVmUn/OUIR5po+hn2j28v/SHXO+iOZBVrrn3WXF3YvLy0eecK7O3s589bD051IqaN5+daa2uCI1Ltu49b392DwdfQKWVfX7PPTDA0GWY3fQuG7SwWxkd8vyI8iyc2Ly7Yp+c4pscIZFOgA9EkyqadSFDqQ1/GPIbyWmDMYeAse3eXbbQEPKk8SA/JsJeu4LXzlNwKjTGJQTV+kdQpLZimSDRmLF812KugnQSWJoqhoxzgynyqxy/3YDidGZ+LB3UKmSXrpmJ4k96Q2qB42inJFmleCY4WratNqsMAsgYBZezQm/cqcQXxjuiP+9cE/wzGuUBS12nxQ0Q20lWh3R0TegBqR512GNR9S/wGECVPlB5yqCLsaLCbE7PNtaXm41U0eS4vw3wdNU8IePjb+KmNtaubd26gGHz6BKO0KAAXmfbiFXvhIb5hC/u3t+1X4/8zfh/+x5OJMWnLF+U6fDq38QJfcnCtfWpBqrs9PQcPvXvodwdT7wPXc32gu/sH4sJ4d/8+gR3B1pZfoBHy29sYbHdxUroWd/fvhGJ6D9Goen5VlPLpCuDcqMdNKuyJCXv8lVN3F9Tl7rWW1oElNG6c3rw673zeDZ+BZdk3xrFlDiifn+W+oE30q/HGzXuAK+iljFbLeRazHLuogOzE/gFWYXDB3CegJWEv73xNsbqXdSBUCyYTfqWPRyn2zcIAWgM9lKmxdLlgZkD6mRX0kSRR/ABjDPOXFHql0RSkQtJiH8ALx6hORYpA2KFR64hrUXe3qKTRu5A8YgyqKCVS9cJDgfJlLJ3JIHivglQqI8fJGoONufnm3ZdJUYMideDc2XzVGkkcKldN7CRjKEccv5fPARtyENMNo/58pmtkMioxHGktQ5lBUNlzn/HUK1zjzRFs/lO5AuxXUyHLzMOWQ8WfRoRyOEdEupRLAepFBC2M9IfAU/p4ihcH/RejJdTyq+QvYbeypny2CTYVZUj9+/Gi2B2fTxBLx7nGqWB0MVk84W93G40FmE45Z4LsvrZ56iya46/ON74/Fdq98xdhh7Ltqw+ZLYTeOn6cXl9aBMHXuPwUie3lFzJC/9ljQFfUKF9dsz9dt88g8Tutkds+hRoNkrhHzdXjImPwQD9CaiXCMXT24w6Ot6uO/x/1gIGdP4/1w/9REEA8PuC3cLmtDpI2lRom7EJhJacjtoeVr/fh63HGvVClXLUqKVDQEr9JgzjzzSeDcAEwOoYkviQcvFEDSLg4JXEkIf9GghUA3z9tlRK7I2L7dndQvp65aEyPEnHpNwcnJjRM19HaC0ZHca7s4jmA6EABXhigSmRSqlH+C3LekJquTon0MTVUzJwgYiTOhDYW6eaL+3UAH3i/WcExCznvQflLeCYB+lHUAppoY6nNHhXQInCAUBwmtLl6pbV01wmYR4Xh/il74onaCaOPwT5zFQZgYbRPJqhzNUP5SSnEpXKihOqXAbiENchWA11mM4yCY9TrPW1g8IxHClcAnVupo2OoSxBIBpXSmikW6nUciH5gQ/QJPqQBOtF/jJkFeTpGWxoOVZU8kuHHaW26MTnZPY1UYtjRmAysxYismAS8fTsDbVPbNb4VAxXgYAUwzCLWyHq81xcCiAwrfMSEUrAjQ3ugRRVS+AUYA2JDmwWJagFD0ZWPAP9EOgfLoKlOWeP9Q4f2ixpWr8Py1/T6g/g2WQdhw7vgjfXbjVN4MKiGW0UzPBjf1d+frKMTDTdqXl8LNaVLKCDyvC6hRkuaNxxXwe0uTyMyVicSSucAxqhpWekjJmIHbAUwFhAxrgp4gHyT4qMLGBDFrdSTQYzJcOneAa9xAkNjBeq79qquLXGlnFYUqVN/WKdIKuVdDx7koEGfqAqxcYYHuO2KRTwFMT1tL53GyPRbz+wXT/HgHikCG2tfdENdq+T0TOLoFjD01+nMAY4TD5DekmuFuJtoxWlnNkpud8C/gFmIlYqlkD4T4a0OhAJDtLgYOlF5t7V8RobSEA/AcNLzq43zUyzEMW7om9tepaSNyZOo9PrtHLXTaK1fRXerapG1X71FEcTMULt49i33hGMyWAlnLsyxHa31+dbtGW38wyPFHJsUYRJi/ITeZacAKrQJWSwmKUNTihxe70YMXXjNB3/SVrEA/DQlCYbsWTiTuiZqz33tltR4tnyjMSroB5BNG1TFsNC42aGoXsRwBVkVR6I1gBMQQTKH121s0JUiDvVDKzH5xJA9bcbdQbSzByML+rQ1AoXkvPJziqsElGdO4/IT1WfqbMT/XXY+AI4rVTrDai8j4dXdrQcJRt/Ww3PcUnQ8ZgnsHk0GLNWhedAezb4sUoPQD16+YQviKok/er9BNInb0IQDzXGk9oHQ2SEMp9Mu1qAqDTWClXAFUlPCvTi3TcidM80V7AiQn3y/0nyyitbl1a+a82fwSNur8eaNcX5srS1vrD6Fkmx5g2ltrX+JQUPPp9kox+dRZeeAYebNMtmwvEelYvvLxeMn3ufvoVo9TJE2NJitbBNYv9KcWg5DJ6+Ur62hai6fSfHX0HbVKZ/23aZuq9+8UKg5e1ZWvvf3gM99oSj3SlTP4huF6mhKRuAi5TpIUv+xangJpkqhIggUqee1qxowJe3zI8Wfgy4Rsgkn8GhdLFMdLVkhhBmOkacQxAH5CY3w1vY/YoQ8NFaiZrzH3Cl6SOrM6RI6nsk+14eNYZypx0uprSr9AZ74Z3cbpi8POSnLgaosDQPt0VaoHEMDSygcy5iekYR/pOKGVlHgVRhmxn6GwhEz9JZPEcC5+9fqdgJVZhjZKJQjOR5BpYQwUTUnH+s9bT65Ltsx1ftByRyT3XdigUoZzStgk6PPEKYSPrW3BWM46osGInVNP44ZQw2+02aCV1EGbpnBq01ikQLlDKnSBzkrMU4cNyYJsbu/nxQ/OQlkBKcmHD+8oxZ2m8guahfxYVaxlHG2cWG5MXOqi1IXCN9V2bJGzUWzuugA0dXPJHdhGyew/BoPse719Xj3bYVKrlA13zR5uzsELbVvzaSkxnhetjnTZAh9INjliUlWGmFC5HnzUjWf9xQztsaZsoZ/pEE7O++Y3kmItyRCKMr/pdgVPEbkTTErDxppaKfYvl1V+IHzyTN1oEpBu95tLPepHhdqEMEjCKKi3x34oBsRwYBzFhkmEZuIklATFMDCC3y4+YCZBS6VISCsypb5DVEI7ZhlrbAilC1jgqNSyFWFnsP6XJMIgp6KhYcKCW5yCgSl59kvL4A619k9F+SuA2hOTCmnoGDgKp8BGVy6Isf/S1QIWtoQH3NX0jAiDvEPXp34j+ryFnvhdZx3QFiDfUn23WuwE91YnW1++pTdsRpz4Y0dygoOJpGSkaNqavU3bX2ojnoVDtwBOcUc05oq12VH5HYoA5sOuSUKUn7aNkVuTfkFU1rAPrGKHdziZsiFoVeuS88QbBGL+WMmGqwpmMx1hbgaC/z12GIxlsxjvZAHBXs6GWGVH1RaxtsK6h+DW1AHAsl2gVEEypIw3pr8BkXTrdubj2bQUwh7cnQETX5tLz2VbsnLT1tLr8UHH3yoMsr429eCMWiitKJyfQswFGYH6IxMtNXw56wxPEcMPGN2vTkoo6ZeY4SUY3JAU6x0ecJ20c2QgyJNF1uB53F7uyfQ2Vi7BeABsnG40Bvg1gdJNjyawWRV6FkUfigJHQRaPk4/2by43li4ubF+w168Iqsdx1j6F+fs+fNoF3813ri0wvzQVQPkpONaMw7ieTPpl/Hh4Bb3wpI9DWeR/J3TtIMKCmtnakcMRQjfnGdYX6g56B415fLuro2Kq0dVWh0iTf4gktDNDKEfvePXp1H3mYaC3JidKTzsa7inmwOqe4e6RSm8FROYmOb5uJNFp2vwbbpaKabcFkOS86HpgViS7K3LnnLKYpPLajvcYjbbPv27+3/xM/dgHKfX2Xhxs7l0mY8WCWoTYxQuXm09fdL868vNy9/ps/2z3XKpVZdotTlbhIx86d+5G5sYBrrnNCnoDdXOG+LxLT4VtXhv89FdSQM0am1uQ75BZmH7KYcpu9If63dHnJAj5j70x7RRck6njefnRL9AI+WVL9A3ARx06ovW2jdA9K69Uy8PnV5fa168Zy+dRnuMlBbVAo065BESEo3k727rFREFCkFSYkUOiOKLMXBRQq2naqyFp2oatDpveaI1+gPfemGb3Us8VvzSiTzK9hZqhXpv3RMljicQq0MYKPtf70b/bzr6l/7oL2KpaBIDZumsKnRBxUpQJgM63JYCCX7ELHMIw48dcTcdUhGpgukgPU2Sh9bijAFdwVFwlHBj1LueqgWR2Fxb2nh+Hg+AvpzjXFT2xGPKKnW2tXyGSRWDeE6t2xOzjcsP+ARl49mz1vp14OscMYSBMZdWOJpq4/miFsKOk5nJHUGzEMWLehfdXXjNSETlIjLLTSptDedyCWmOsIrlCvIwi2JG0LJWSsPUFMtWImREyKxnhNus2zLVU8waSe9862chDNX1maDCsRHzuOymMuJgPD3WRGWSvwwOqqC5iDdOLqIFsiXdGDdciqQ8elIsZHNHyHxjDY+YlFFhJxu4CaHYMiPZtmPhUcYdDbccEw49+THB0R3N46jPyhw5g1qNZEQzUQcxfD3ZQpD9OOJzAWpiwQ+pm+VSHldwSEvqqDTjdZ1q5dzzBIYjbilJqRQhClM6fLRUJBJNV0bidC7CMbQNEh0RNTDaoQYe939yA1LUMQlQ0KyKpe83GVfpI9qEUkMnKnje0Q1vkUceEcW0A5FHgbmahYwlI52ckIRsFnUs4FoeS2CukDGPR5xThe4hAj5CFehYpMOFHePeZUM7Es5JrPYNY+A2LdAMVzaHFcGyOZQabzOFBm0vqdL/OfjxR3uoNG3qIuJ3hRw+ae/CXcwJODso3LVDZVEx0M0m8HtkKMomwJnzOD2f0E8AMOAdYoBi/xuzN+0bcxwKCrotqAKc3wyT/b14yuF6nbfy2wSmKIUt6vNFaA+9HHOPQORvvDxjz04DdIwNmJhsPljdvHoXmrFXXm+eRmMD2uTW1+zpW8BNW0sr9tpFZL1Xvm0s3tl4fhePJHyLwZa44/n+mX4MVT8WFBvDBGcSNwETB+O8/sn2RdcNffazbzfWl+y7szwrMZK2gjOzNhYecfYA9NOofrWW72H+wFfXYFOO/SXPevDRCCYJpyNbsNhKvNaMR4B2EemOkAc+vB95D4bsOUynLjr50X8UOuDu8soyZxRFpzkJSp4Joy2oxOkMmcjw3CIfVKGdK9srEMaN283rS5ywETEk2ByLrGoQAErBkySP9nDPVqkbe5Mec06Ys+jL8FHDIUz85Jsl6N/sk4012NF9Df0TSKqYCpdzU1J49Ezz1cvG7B2ckxc3MeRv6a+gVgKuS8QlummtX6XUkufhE4YN3LjtR1byyxCekUPGSA8FHhAjnsgILfvrkQ1oW5AZRUaKYymYmWI5g8igOa5ck4UKQz1cwKdoNCp4HQVMbaKGk9wLv3qTdUrx6mbDFPyNBHxvUh1hUtAoWcLcPXtxHkN8qKTSDNzCjBWDmhlJOyRO5/OEEfsTCLCQv5jMuKExVL1hFYAGDcuwolovx173qoO3wEzfEfCJNePeXn+XGEm17ngGdvbl5oWrzsCorDsqr83Oyb/gjEVPxEBtKbtaklIydB0ZZ9bAlmVahA7DkzkXgsfoB8rpQJzhoOuUlntSPqO3m15goEmtl5Oe9qqp6gXsgr+V3qRsh8Nhe9sjmQkjKeVp7i/EnDwqOSj5ZiUd6FvFIEVaNKiiNEdqE/1e1VE8tSFfGLqi6ZSWR2Dd0jhEKKqOygVUQc2qSsnQHAWIkhrwmSc6MagfF4QtFLrhy15Q7WdA4KfL5502BrNcJoFH3+rkCaxRHxNRzBHk7XQCj4S47UDLW/UVM8hAiD8SoDpbHH/TxCJQjwJb274dlxu99VDbFwVjcNrO4RMYWFOsFjIhtd59UFTsDvPO1jlo7qutcCGu0CSi91T+6tAiyP2UKulvOlQAnUpWD7vdeMfBr04dIgZhubMkmYsna4QXB30AKuVqYRiUXWemWZJKMzpnNQECOVIZMboD4hRCZOgmAziQWEjmz6Xlw89tNBRum1+JtzTD8re/IZ+53sEP/XUSdZb+7j0Osts7NO//tBWAWlDZSNpZFWIvHhEjjxt3B5Y+diRlgXwF7cFBGNk79do/m4zDu/w4rLwOrKYOW8e0PVtE+JmlT4YPF/MUBqSO2DPHiTDFc6gDk0REogD+kNhDJ924q+2xZW9ai0j7fGKWDJI++IsxXT9Ml+wJOiZvjBlbPiuP+phTAKQWntjKCz46xe5KdG++eNy85R5dHJOn6Wl2QcPVQmHku+JYCGcx3NN2KI/m3jMRb5Dpb5Tovna9BhQpLTDHkJQfPqZKKOO8jqhzi44E7MK9XRKnpiQGJJU/T8faiAgm1yB/nga+A00i9I4kpjK3DcbfAmWmK3jFOfUOK8RKOnnSULdRydVDMiw/guuf8tlBPI47OkAxCPKoTetjpZLCJjgRvuG3pjEVooIxqLHHZLtKrHK8i5o12KuETm8yHtudJY3L0vSpeh8+FhzVS8/0RfWl3GHdXPd6ZdWpOtpWUVmWQVzUSXVHXxzhEqzFs85P15dcYzMn6h81pfqDitKbZKWRftblpS2yaIeSWkFvv3mMbZXckWKtzr0UnXq55U7+mD5urYtMD5hOGVZwglfFQ2CABhFaHneF28hJlnEvQiDUuO7toD17EbogUeDFE/uGRDdXWCHG7czCpv7B5wE9xbM6q2dwQ6BoRtseWVrAmnKtUIzWYJsoSHoP92rENNh48rDx6VxStJZeYx4G5/6NbBYngFLGbJ6d2VidResCh1WdmbCX8HIHPZgLTewytHy5dXrNnnhCEfSySudOF7r0GY9q/7BOo1vMnv0OOcSFp/bkSpcOcz99EWPefjq6Xofdp9MNP/Ei3qFBanJFsz+jzW3z6rxD8nzxjT19y777QHjO09ovnvIp/WzZtEYE22A8XZN67Fb65bAb4WKiTBEwg042upcDrTvk/mx8Awv4iI1mreWLOGEvr3uaVvo7otQP64lDl6C8+zvm5rh04g1kfpb3Dv6eIg6kqLcnL3MeA7zkhkyeRlhz/LmOMelH9vub1Jl11y3WwWMWJPze6OZ1xZi27tI5r73BeLTOHmQJNSyzvOqnGtnLr71SUSued9pmhlSUYL9tWCXsOsqh6wRHPUIPA2IUBp0k1W8OVwis7gtf0Bws0nOivaFTU4FuTg3VfCEY7bPoObPQKUQAkYSng93l7T7kri1wegQ6upDCLMtaXwfjAzt5rMVqJZXJlT1JseS7sF4gNnoU/gvaOF6dohyJ5OpJFY9qHpp88UhKZtlSoPvQP012cb7Kx5vggZkJMltVNcCD5Hyq4h0QR1U23OJYqphFY4hSkGGL5XUkKpsCXzsEJXNx8sSofFPwE1NQ+rxhqrlwDDTwUas9lxR6VGQT2APsPf8czCV1lRNlhWPX7pyfyVdfswc5lWWOq8CaqozuGKQ9v/aCJ6JSzBT9dijMqMH7CJpCla9AzhljiRLzwasKX3lNpUrduajnRqMgPIBeYR/1dKR4qdT4Kl4+wmmH+cKn+2ea1y/xvWXaXVDodsd8xSSaQKQ5niSZo/TiVTKto5MJfQmXHzBw3cyuhqpQbTRn4VUhXiz3D7ZrYZzJUU5/izscJGoQNfgbhxr2Gdkx7IcnK8hiPvFY1BwoSvtkbwzGxzz7FtQhNLxTi/K7X/kDMT73mahJnokKpSYDatpxGPiiM/CafiTVZ7ylmVNn+//xoDRHxuQ572SQS949NYIjwRnpeO6mgjctUhAw/kcjZ4rWJYlPZtyI9/B7mx+XY9rkyFT3YpgpJY3nd5i8E9ATnQeE286/utHWAaG7W8jCxOo7q0uYUGXtsf3FrD35NbvHUD9eWkHv0o17Mo9MJ1eZGjmQJ56kD5nhWIomN5XCsHZYZd5y6xbpiszfUK14g34p57EmAs1MRIwU2fHvpiWXpTkhKLXusVqEVOpodgG2Rx1zJiaaYWkRcVIq0UvXOELWjzh3JDDAWIuh0AV+UGSwMhcjMOLKvqbIZBLH4Ua02GVnipQZNgC23zaoISnwiEp/ROzcSrCz05T3w1bjnLXF9H8MAqFHdTtV3ZfkadYNXmrRB4OMN1ogufzlrH+cUcSXastLXXVmA5UUZYOVKWpTSlPz03zBRISE/6ZYzUeDDKxagZJ7QBXi0hxXQjeLUcKO0FsRl1uLvj4x0B9mn/SD5sv7aFGcXYF3gry51+2X9zHCd+IOCZin9vjV1tkn9spnjeU5sasfN9TaifBSsYhaavtNcyHkjJJDJ+RfKQ7MQobPQtewNgbnjeYqISwD+oxmjsbuSqmv8zHXsqq0HDTItltX2ZoKZbw8amwE06XJTsQDQjjs5/dbTydw5AsrzUU0S6lLS1HuzkyJ9yrl/I73RGMKJbI9N41FTy0Av2p3YRPdICNxB40X8IXko3sYcSDWH5FMIjU2YhYSvsv8woEp1bJV0nRUM53yO6PZDorGWCMK0dq7LFzArnRj9TxsSXGbOH3dPvsyEA7MpPTIa0F7FJ3RHrIXlvdLhTtCItkbCnfoMWlub4h/cNNoj2EQxECnj4pSdiQ8jm99OxdYNZeVpPVL1u/wId7RdMtFO/ZDFdFiFcKdWnXULYxrpRAOJHqg5FzBQ8lh9dHhBcQZOvcx4KgUXkVbg0psLKBLZV0rxsQkBzxxsCeedKfbLKCkyhMXHB/gG49CT+puCAcTdZjVOwnmUzK0R5EJatEeQaIxRfT4dzgLlsu6XFBNHi1M5zlh5TR4LSSsiJclwwAigbqYP35HXRch0TnGiVP9bhJnIzQ83NXZn6GcD320/JliUCgfM1dUh/xSmGULOebMscB6Jh2xRR29tX6ldf8UyAVRC7lCI4o1w2I7NdAnftYfjw1k68KePLP5+U3OMC2XTs/RHjzZ0oRVc0YSf2d3vU/bFSR1LwLMC/kOhOh2JM+1WfMMyhiPdj89NQjAajAQx0LcByhdgykI2hS0qby+GEV34wNCoLF4hzOEBgBS4hIYSbUC8qQQQlGgQpyHMZ4tL2+3bYsYtWIpOjY04OnTb80TQ8V0ObMPvdvlaqkS2Ktn3/ImEoMPYes5OY9WP55tRkzehuFZ3sUrIP/hm5SrdXn1sHbRsDRoahcNY9D3v2iwQOc97YBkoDaFn6AlQAkLR1R0tgz4rQKjjPY1o1ylIA0yPm0pOLvnTbHTkS2r6B1joj1WPAMoGNTUFPbToZRg7zVtG5xoIgrK6VCBlW1fbLk/UJw7iFEEZOD1gegUFN4tDLwn4KikmUmlsSd80rVSzhL3Mn76x+hPR6M/zYifvh//6Yfxnx40ZJSNNF4wy9QCrjjIStpgnUgDZROJaAFQHvMHF8X4edcKqqk5smzw6QQMGS1UErDPaVdBXQOI665VsFSnup+g+llMGpBatzHbKh6DnftMBVeeYqrEw8bdjlXxFeghac4KMqtjDmHaTEA5MxMiU6qyosaO5ItDIWO71/YV9pxypPrt7KcmYdQFXbZ2pYOLGYB4W2w3oOI/b7dADBk8sNhoxgjrS2hIuxrfcDa3zBOITAyYHPFAx9IGSmqnZfM5v1U25aQTK1di5dszchcvtUj5Eh8boL61tRVuMxLjZi6G2xxpK6eYbAAxWjxm8h0OIcO3GKhvS+N0uC1noiypb5ApaIb4LupaoTbLdZuo9di3McNeCO8pSI0imaYKFmhlpYDXvgVsSwjbevaktX7WXrmGkea0MHie/tm38LKxcBM3oLRyIK05Ll33rbFdCC88A2l0/bvGpRWf+tl2tAFXUGXrl5Pi2OFBv0cSwRye+fToUCYtrLgIWYOuNEAX16AUA/xbsk9+kN7XiIywkMvqXIZlnBQyUfNJITNTnJRnBPEHeUnxB19QclJgqqnJeT6ohe/JRIzXLJzUlCN4z1oPvCQvuNRl4Fk/02jhM4XANldvtpbgsz/ExTgZjUY7/X+8039OGi4tWErgaoHBpNEa/zN+wZA+Tifq0Rfrg+lG2oJFDC2ZURsgLSotGFZbJICh2QAxOBAhRlR0E1SCla8naqSj1znGkPK48gU4GODJnNGSZio9zDAianXH06LRs0pcUymraiRJIzyUsHbj2km6MvPwSf0WNuJMhtCT26nTZFlduCZqDLxXe9cbEb3QRi9eOeMKZaek+8otaHhPMyk6yQLiyjgOIgP0K/MbogT3kYnBfSZ6kI84bPrBI1BFCvy5g16POd2Hh1VZLfgEX5jOh+AAEm4MVwd+GmH9Yk+cWF51eYyITRU9rn28i5hRpXRpg//pKvqpBPovpTpOZ+bpchSAVHf87Blg8EF5GLZ+4PT/U1/5P9vvwuPo4HVp83Z4bhLXV81IGLDh/Xm/14vlHlQGzJBHleGXZOWeM8v4mhm7PLbsQQojiv8EaHPAqO3zqJ40Zs6CYBP0wVOWocrVCPIFeSykHi+PXvVNe1ntPgXPZlYqzMxPva2Tkb/Hn+USetAWB6kb1Om7FkPan3TPpshKbQFKPF366X3PRLklHRDqZGTArLMIRo2QAmQCprxbdweSqmSwk8hX3oM/3D6LYwqCQyAUGQlwVBwkHUCR97TSXTmc2aRx+SnxPlwGNsCAwG+PjJTRd9gLf/wd4qSKvqPvAeyzLrUGEkDtB1xG08A6fZzGg36IRRbs1cnizBwfCZ6NlAk3exr5xjlmzIlSWpiCbU7j9ERzcao59aKxNG0vXm5+c186wM8+2Xh5vjF9b/PC1Z4gD6N+T5pl8bQzx+LMlu+Wj1Qx6mU/PpVDGdMaBgTBmolUKlMcTqUigtlDxSynKDlowql8ID22x63wvpkv/UYVlSuMtyCk0rINxDRHd4zIzKhWgiyCTgwPHmzmbIwJN+k4YBQATxisL6IHlDLeCqdAp+bk1ROigLwzYexwIDm3V3BsHN5Wc+aqfffaxvNHG69voNvn2WeAYb52otk8noPvE38eMwu7Ym/Fdw8ZXUaKqrCnbf+QkRVrAx6UqTOC46Flx4khMm9t77vbZYbUqXOsmgf2bZDCOyMUMBlxU2Mm9W5iia6d1FJnYFa5bFZgTuM+TOSOqcX6FCy6HfnVLU6fB5t3pGgtpbE7HGyx02Cwn+jrTnBwl+xlv4M0c49wVzQ1KwoyeKJfXXdN1n1gNLDxbS2f2rx+Rrz9851caKpjc8QGDQ1NtUtxHPSiTYs9+bk9ecZF17bbc9qBO9cXuaA4wRPfjkinc6HDFetY3zBeHDXT9Ua+bsNw7mGCltKUgC5hWJViGVS4ctVtHhMZBV3ZxNcxcbdg4jo2o+6C0cZDsbt0jQzdNCxzVuHSX3jREU66lItisiYFRqaFwpuzV1+2lvAmY4DQPL9s3zltz13ZPDsnOmShCuASTkCsB5NUf8eBtnjzCT/wwhGZFkfiJqfhBnLESyy1bJB4Bcj0w83xzkug2wNlw5zWRDatiOCakzRZ9OOSq+sWMLNNR+AqfrJ9RGjAxoP4oGXdAp7BkbrtzE68TTem0yVlA53HIBM/BfXfnr3owJfTuDgFjWoR6O3UBTI/kJ53OVApWox99EDVAA8PkGPaZIwgd6ltV8c2oL9R2MjoNCyN4C7ak0FOpjm4+8KFqgp2gk0xb91JSiI7RXghBayet+9+5wm9DuALsBPqDhUTK8x91rgx37iEbmuYkM1PvrLn7m28Pgf/pYPc52SgHU1U56bKJ6Jo/X9Ta+wJlZwOUJ8EE98drt0zdAq1lOnHm58/7tgg7yW7t9eYnvYtCY+xMTWOBk2yXaJlbHIeGIr96hYweNmevAmixKm8sWFLOUzUNo07oKfHIuNwe+Csz/7p5jhjOOqZt5qoerjbTHwiw+JgYLiyNKJR0u5uBdzDgx5jMzcWlFXPVbnUhRCbC+PozqRb3bhbmIPL1YcmPZeibF6b23i9jik7KJunPfsEVI7gbKDGBx+Kg5VqJldEiM2LM5s37mCAyJXXIGtVKr8cegZy2ZxZdoUTSJWhItkEQ6O4wcU9LUe9jKKdSc4k2ZfJi6UmG7e0+LYy4rwlxSbpXV6kHW+KaacVrMet+65cbDc8RNxabcigLwU3WT6RAhrSUapj2jmKsFGBBSzTZuyJWfv1ZZaxRLQk/admNU1fvxZP3Q/pXtLc0/GWQG/CRdhd0K203IJ7/CJ4FCVaJemR10KaB7wexvaUAwyebn6hDtNVMf6g10m+7JrOc9WgKRULy3e982m0mh5v65bRcypvaYFdv7u9ckYmp60FxeIrQE7sfb1DKtysoYfcbsm2RLgUlKHhcMGJK2EzN0yOLw8H3pbYn3RzccAc6HYTKSQlT77mCTziVpQliGcHYQ0kk0GBr/QtOPjVn+fRvSXKG2OLibmHisV8SE9tinPy5sShwffnHS74SyJu9KKfhPwDlKkCmuXsFI2FR/J13Qim3TfiTMcTP4oEuzAKmZ+DQWJ+jgG/JSfcXfT09OQwjz5H8dJEp1Joc0ilDGXWRANEz/8C3yu3Og=="},
    "deberta_sst2.py": {"sha256": "fb581c229e2d119a193d627c86c0d40eb3a894538b32f303d011f89aeb9c9ee0", "bytes": 23619,
        "zlib_base64": "eNrNPGtz28a13/UrtuhkDLoURMmP5nLKzKSN02Sa19hpO3cYDgYiQQkRCTIAKFtlNCPHD0mOFDsP14ktJ7ETJ75pLNlpmjiW7cz0p7QCKX3KX+g5Z3eBBUDKStp753oSCljs4+w5Z89rz+7PfzbW8b2xSccds91Z1p4LplvugRFN044de3l0gvVOnemd/mjr3kq4dLZ/Z/OH+5fDh+vbt0+xp+xfHzn6sjU6e2B00vLtH+6vbD1Y27p7r7eyuLX5aXjrs63vPg7XP9z5YqV/+XS48V3/zOf9zdP/XHhjZGR7/VZ/81Tv4u3e6jp0Gy79JTz/Rf+9m2z8MBtjhw/Cz8QhfBwvTBxkvavXfri/1Lv1yfbXH4UXToer53Cszbd4LztXzu5cPt9bu9c79+CH+8vhOyvhmZs7p26yw7/MHzj4X7z1ykhv43z/81Uc4wDbuvsFbwIl4a0L8Hnn1INw7WZ4YWXr7sLW5gP2+C8nsCHAvr1xEkbYunuuv34pXDzbv7bef2sRJhF94pD3rt7onbsRvn2uv/lu78M1HHHr7qr4sPL29oN1nO3Sn7e/OROefwOeoUZvbaG/uQTP4a1L4e07vfc3+Iywi6sf9S59s/PGJwBq7+a18MM3AYe9pQv9zf/54f6Vrbu3dhbeCxc/7n0pQAQgttdvhw8u9q581b/3PUeyHCq8ci9cv4w4u7sKs1QnBXQEgo4wFt5+u/fxt721LwHT44f/sfBujPllJDlRE2bOJuzRQ9DVpBVUp9nj8MRhBkoVCqx36wb2BfhfPD+81YEJeDzAACdU+Q6gHSo/WbOaf0SsXD21s7i6fe12uHieFYzCOEAUnjzff7AOpGXPWXO290LLawIRRSWCcCVcXgUYHmOIp1s3YNaAht7C5+y45TU7baQHE/+2vnszvPAW/7zzwWq4dJsVAKm96xvhvc+2PzkZLn/Bxg0qWlvYufg9G594HFlme2O1v7wIxfU2sGnv22/DC0v9rx5Co5ERJPRfL3JU8hXEavak7QWW6fvBhNGeY6Ojs1ZjtO40bIZFJrw5NStwWq4R+LPwudUJRmuOJxvuqSfDMJSW/M1vtmZsmunPcT0BDf+5cHKciAMPgl8AW4eJvj95mJo3N+p1XBoG2W9jk69nGKO3/G64ent7/Vq49I5gu3Mfbz94wDkZOPPhu+HiPaBr3OfW3Tc5pyE8Dcs1XvVbLhvwjy9uuVBOA7m31x/2iZUAGn/sV1AYXlh9YszvNJuWNxd3BEtbbQBw9s9901s4iQDTSlSWt0CTXE0nkb6XvsGa5x4Aw1AnJMJozTCndiIzftuza04V6esbVSBwJFMWLuDv9dO9v8GUL0NT7H/tIxCVvY8X+7ce4uj0FYe79cnO1eu9z94AqMQI2Bl8kdNr1iLU9JYXemvLMD7MdPsaijIu+7Y2b4Tnz/VWlgXMV68hpe4s9hY2UTJfXQwX7rO/32EwPKAg/HYdhf9I3Ws1mWnWO0HHs02TOc12ywuY5bqtgNjWHxmRZd5U2/J8W74DhPJxqiqfpi1/uuFMylcki3xuWsG0fG758smz3FqrKd98HNQPnGr03Z+LHgOnGQ0eeFbVnrSqM3wGbegbhpXgv4RDjTz/4lNHnmMlpjWdqtfyW/VgTHC/1GbayPPPHj364lGsNB0Ebb84NjZdH206ntfyjGqrqY2MjNTsOmu0pvSmP1UEAL0cG32CvdBy7SIJm7bnuIGOsBnwsY4PuvbYM8XHni8+dkzL5Rm0y7N6o+NPl172OnZOdHnccwLbRATpCH2RgM4zWI1QJzlE0GwDhFjLOO4E06ZrwRj0ik/sF0wzoIqWk5UN3ndgnwh0HMCodZptX6eu88x2faS15Vcdp/S01fChzHFrthuUJvIgH+pWpxGUaKLQ8yuuhk2qrZrjTpW0TlAffVwZybNhHVc5NDizn7PS4H8gbBkXHcNqEFY826qZICkFTgAK9jphhlDSAN4o43qrcMQAB4NEYj7ADiDaYEo0rEm7AX9hwYEoWA6XLgHzM+gQdQ4tPFDAqAmWwMyARQfiCOwPbnjwV9CqKOX+enHr7lvbG7hCt09fhuWPy23xa/iKywYH91rHfSBLuUJvSBjWatucmhmUjfrOFCDStY83HNcuaVqOWT6rFyN9hRO3PegPlpXxFEzxKBXodSRJw2kCQb2S9kogcI//nDqDVcq6mpw/DKARArR5w/H9zqRvBzrv2Kg7dqOG7OKzlgdA5+KhaXjL8W12bM4P7OaRE06g17UuTmSehd+f2bm2Ga6cCS/8JcY0ammO6627nwKWFbDq0L+TZ/DjAhY6TduzAluAkRqV91BiuIK8soC9kkvUgVnyajhX6FIv5Nl4qp/dZ9D/8kvWdYCZJ+YZSEy0HYgTwnvvgSHLCmPjaOBsbKKN0KWx5rUUDMBNEZTwAjAiXLpnTAGGqSSHaAWyohBw2nqO2bCwmJNEMnCMYbWBSWp6l1oVsWugWkTBIoMRoreK7C0ibJEjY57D59kgtl3qV0gVfDT9aQsMax2fi8qaoSUEHXLcTcOMhLQ2RAPeKdKPiEftowlMG512DSkJePXK+wDufZX5VwJ8lvBGBQQjvrmaQQsBRKKQHAnAp41p+0TNmbL9QM9FohYkwHTd9NsNICL9xmI3LQBI+ANUIMsDX0p/6kEUUq0arlO1VNfcuc5os9EYmwKRiMsGzSH4S8OV6DcBaDkil8oDP45wKSafT7BGhPSaX1ERAXrOcXVQvT7NP+i0G3Y5xgIIhxgZwJFY0aA2Jpp0injhwioSr6mKOUUORSyVR9HS6ng0u7pGHXZTDec1Po/Am8sMlqSkRo3EwrJPVO12wI7QH7AyUBjaRTQy+w/e7m+u7Vxa37n+PhhOvdXrOxfXwlPnexe/RoducTW8sIE2z5foLT7ztHnkhadeevHZF15m25+dROdu43545gYYvmD+hTfQVUyLy5YPLDnreKAWafEqfWhp8QRav66BD8qe6UxNgTBnT4OuQysWzNzILw0/vbP99Q2AphvMtW3dzhkmaWjTnC+yrj3PPZfee9+hW9rlBsc8Q+9n42JKzsSwlRNwVQCbvGGm+gm7OquDkURPncCabIA6LycLKiD7sARoN6tQGgUmZxy33oIRFGon1gd5MoyTD/gZKAoCrN0Cfoaau6ETKgtzQhPsnuAuHFbwenXars4ozhLaKqroyrNoCeB7pP3Bkt9e/zQ8ewZMfVT0wvpG7ZQi2Tn4rPjEqMvR5ZQetXBgHtwJ313llOWmgFT2u64tPqmuRrOwa4AVMqsGrIzpenZdxLPedXE8cjjAto12Ky3WIbyolYsHCoUKh8xH4xEMGD0hufJMEVKxXAIogQv3Wnu6XhkZAivawABqE4MEto94UBBQJKA4kyGToL4Dk2q6DvzDOQW+gwg0Jy1wH6t2TSeCJFllhmQtCHPbrtHjMNsxEV1aXAGvj82ISFIyDnVFxIzufsFjSeBTgRnJvUT4BP4hGRXoJaoGInBfSTg4xlH6oyNMnMqujV/LXow0mgpyWoxSxHiBo7Ld8vdWf1zUdxDhCIA7ZXCs6TAmoIeNjbEJNO6VL9A7fhmVX+UEwDTo1OtYgbpLKEVetKvRP2EwjmEeJtjV9J/sOI2aiWEB0k5gLJuCuMNI6Dt/sgkpfmR1IXJ8RA6tVqpgiHWW18hoi+pyNCFBqA9Uz36mA/y6ewcIcOwGUHMYFXug0YsJuxj7o0/YbzFt51JDwJpW7zTAYsiYuG6eUWwtzxroKQj8cNloYBtTfI4LGl6ml7btmXa7VUUDEH1yo2o7Dd0Fp4la5zINwKJu+4nKcRf7lbGoxM+2t2GBm1YAPfhgnsGa7TbBplH7gBVPg2TbZv8hGgm9sLCmbH08r8CVgQWYfDw3Dy5kl/qfHzA5kEK1TgMFIUjNdAfFqXmMYGqJZmjZDyCO8BCQiNlh0Phgv2IT6CW47ImItwdOeIArowZ1WNedx8AoRiYvnNWyg0VMko9pJ6xIo24fl1zi5iSnQBlWjl6oVZZxUOVzRjiBeKfqRFz+ZU+U5029Vgd8IA7cfvYq8J7ogi/AVxP0FeP+SFLyWhgPTVKP4o6RE4bKkRSm20XCzZu4Mrv4M09uAZShUoI/8CaIBgUumf6kz/BPfjjfag3b8lwwREx0g1GfAaI1ooApehfk0Ahi7BH/7tYlj3absj45FoRQogj/CmjlS4q8FIEZ7Fw87ta/IBvUFk/QxXHbmZoOzJpdtebgAx9JKdutPyC6CWp8KpiWLeMSNAXAP2xEX/Al6d4ixaRP5Jl1qxq0POIdoef5hMVL0ArQdJQqow7GlmIuqrsFDP19dP3Ds6vJfQLQ4dg9Q8u/wML7C+Hnb4ICA3vxOas5WbOeO4obEP17n9GGS8pMFJ0/AS1B5RMRYOELGNNGHE2DOBuWAK+jzhxXS8Eo5JlO0wL1jPWxrlhHspg3ze0egTsglTGPke+qjKutRgOdfd9+TcYQ6AfwWgETqw0WrFMTKK/Zs07VFr6TjM+2vOq0CIzVgmkhNNCWU3Us9s5J7XAljKu8zDuvAAfrvPEo4w1TLbnGbVr+DDYtj2MLMQL0UthrB5IUBLIBdi3IKh3ggYmhEV3i5Y2WOyWnWhIzTvJ8oj1C9egOZMwDl1kH0U3cT8HDPA/0+BLXsmmedXzbxM0pYQyQFBmGfOqPZLQI7rS91mQ6asnhc1vmlGfV9FzSYnFiUQx8iGgE4HIDxo5jZX5eEkVyETQpO0UMwcXNKumZJQW6ApnVCVpVyw90Xs8kpGrVTs3SkiimxY6IsV10e2sliaoB4ULw7R2uzBBFuuO2OwEA45cIfisAOqJHYuJMSviTM3iTpDJBdIKnHaA24UDgPgPxOtXmMGEcqOY0S6PjuXIxDxa6UW139JwRtHBR6TlJG8UUBQkCpr9xiLNsG6lAg3G6WdUqKtVOU+d1YXXPiZp5NoeV/wQWKnUYMxKardWAb0KgDEFi8m8JcQt95/lYeQ6S9LzsgLSjnvSr4m2KDPcJ1ydqlMs6xKKN22m259DPddvRJ7dtDO5A+MfPUtMj6PLG/bUtn1OIE6NpuR3QYGkI6BsykFrBtBoNUYlPmPv6MD3db9vVIkUe8hRtTPqbmVgFKk1awkFrxnaB14VtldaRNduq4XZAkaupeGmDTdBqNVLhjgx61QLDdcFudWknEjQCYPLpOEAKE3F9YI6m7UVB0idhUT2PrP90yztmv9ZBd/43DcCeU3eq5IyP8NgprTlgNj4Mf9X56ssJD0ryBWCpzM0iEcPHHSjutaoxAyXYCbjkDA1faAyBL51aAgI9mBNBU+Jxg9hwKPHhFNuikitr0TrWeL+cvwUI0kkeMD4XRSoIBhbRG34AxT7si09hRVwGPOZfGIlFL3T4aEQbSCQTlhoxXIRIbhYBO8HiMPk8SsIxl31z8S5EDEgTISFzfLces0K2vr/aWznJ6m1KxwiXwWl4G+OQUqay/volhixH/VZbzaaD5vqUHYAI9Lg+Mqott+6A8tJMXsHEfQOQvTjnnOASMAERy+1YXPG2bcsDEzsAztPJe8Ytw9c6jmf7pG7IRmobLkhHFHgTnBqgi/4jPf5KdthqB04TaRcxMpUYlI6il7sadYrGNDdmM/YuJ0misDKf34O7ypS+5bSy3YORN1/ZS3cNT/B90q2Q6w3NQWkQo4LgK5L8hEpeTkL1HkQQRDgFafSAuR19MqTtq0fIRNWCZQxNxNgylwAI81RoN79qqQNYzbbxWyDXMSrWpTrPaO60wAYyg7xp29aM2bSbLW/OxJwBH7ePhsbcFLlEtvaIGkzBgCwoQhQFOYkixT2rSDFY59VtX7WQjk9j4gyKpqSF0fL4Zi6pd249ubmkgaMG16h6LrMhlLS9JMADjJk53Nzlg6p2loAd9L5tNQGaaAIjKefcDgSSpOdXkRYB6grTnrW9OWHCHypIr2MME/d4vWrHmwXh7KKccJ0AHhst3zePO4D/42Rt5hm0E0YnEqyDwr2AeRcgUtoNWM41oD5lUVAGhSLneMxdlJC/BIZRHGzDAjVawIH7RWKPuDrdAvWDITNMieD4UPbRB1isZTJZKzEZeBe7GK1zEWsLF6DMZfbgXh7lGYz8X9jB/74NjGSG9k8bmGDjg9ETeK32XMb2nYvnQ0kotAni+wawSlPPJDRQPM/xOSvxrJUU4BGjsV+U2Hhy5UnhZMD/wqVBiRG0TGhlixSc4aE8LqYM+qMjkDkD04xAotX03KCaHZcezFgs5ga4hWCedQKn4RvVhtMmqAAcr2nqWZ2Wx7zEgUMhrw8bRkLD9+hz/wZOJF3F8pWxMk6HkXSwTUCVICKtycdS0gMWLZPrtVQScZrEoE3bck3BUXEimFHHcl2BiLS+KmDI7OLMprmWqw2dS+z4xgxgtX3aJlFkDxsF2TRgJ5jUOQnKfRg33FcRuSU4p/mxLk2JIo5iYXSjGRWNg/V5xrSM5K5rcfpsN8Yp2F9g8vnw4+m5cqFSNCZsaB9++xVtIkuox9jhQtEYh67DpbM773wEJt6gIXaun95ev7b9/fvbn51UGxMt9meDS8lOtSxtMbOHa4okBcl1NOE/wGc6phE5RQMtcdxezIQ6Hm0PxbGQOBg9YBeD9FMc9MU5iDgrmGIAdMezqhTVrFbncz+W8EhwkW8rU0tZF3oimmf3mIGAW/fe6q28zfNkf7i/nOLXrNITqJfuIlm4Krs+kfwkQo4DVhjXvKh1/aAF2KiZsqG2t1mHS7e3Hz7sLX3Ru3UdU9cvfY1exdrNJDbCk2vwPTWtSVC6MyMiAoCOtW+DX0EBj4FLLxOG+H/CU0JJEUuhQKOH8uh4hRvaAMDPSkQCxfz40ewHVm6nEfCcDKIZ1edmU6pFHk0oz7OrGK9PRITSO/NtkZSnhIZm0V1IxIXSCNJcEzEgNuLhKSd3Kswa6I54HgmiYjoU7UgkSjFxL9M/7wt3A3mtqCmx8RhL9XCApH+Sg0j+ozOaBV4aCiSFbe6ERUaqhmSGopjiGpEKirhFm+6O2E44wViJHjK1VOdkajJGhRp1OiErWI0GGHW4N4aSd4Lt388OgKk8kUWUkmoOfUYZaTIbTck0iwmf582grE2PrUmz3fKdwKFJcrhey7PDqWy0wbuu4O9BL69JFiLeEXwk1qnKSSLJBAMVYrUqfmOk6fLCcKG6U1UD7W9gZT3j99nNdjBnVi1oqCeTH2mpyPTHDnAFSKxExG5AishPDdm1wECuOZ5M1s4E8Gx3lo+ait0hXHgAoiR7AFprmOevoRom14t2IitqbaM5A7862IZgV/siAGafAKjN1oxitIlDAiZmu6LzK8aCEdTTEVzCz9hzKEh1vrOp7GnKDc18escyuVsZbVPmU9uQSUQl4xv5xBZgtOMXyVN1AgZNMOFktxpon1EiO6ZO+XqiPk9pRL8uk6We0J7QC89QEwI1R5kWsfdJqhNDwLLiDNUg4sxwrTKDrI8YHJgnWE7qygpaa+H6Sm8Jj55tf/vX7e8X0RqIbQQYqLxPCvN9FWEvpM0BweZQeeRH2AGjaMQpu42gIhCJ3f37EUiiolArmj/jJE2BWBYoZxRUnOdFh7mRPWBg6XbKXOCo0LIpp9SnCLdYgYWGgcxwEiEcCsjEGTJc8A/MCJORHMneUfRLDU8PgVqcdetGYaH5+HDiztXrYOiAlo2znJPh5JzI94wNKsxFoAGk0INB0Go6c0daV7SC0LwC2yk6Pye+xWtPNFPcBaqQWK1QR5PSMU3y/ftBOkWKmoAHVdYB0QJc0NUKwnxIJZ+lZrenkGc2A2B8YOfjadRldB6HlSs6irA/ghfKUhumYvwDrA4LE1J4lkP68M5/jz7WHH2sxuJTPIOSnKWJltwm4htEwrIaqlWihRrv+ZTQquQZOO0DEwnRxYcqc3OlQsteFmUMnApmHhiFQ+DXyVwbUVUx2ip7E2B0BHLr7pu9tWXWFd3sS48o+JZ7vDtXP4TKOx9ciDOecTaU7bx+i+dtZ+TbfxiPlAw7aAyOQBO0LmVFm56NqhLhowRrVKh7y7/FlSUjLVyOlrS65TRIfVIObmm3DNyJQqEwNNgfHWwrRU8Gbt1ZgQlw6bny6GFoXlQOyfzbQlrkr1/u8gblfTSFWJRkWH/gbqT8lzHikoGwIcbcILwnB4x2drMqQ26eC+uYzjkgwY12q60n7OZc6qyWYiulTnKSgTPg6NtuZ7iIEuoZrj9SAZ7hio9flbiEisx1aaJn7PM0iT1+rG+anwsb/BFzpnVlJqoukBzLcYMguY4/TTKw9GgJKDC3F14bwmcRe/ElQ+rssmIOyc8pi0hVpHjIQlYTTi90MxaVcV+Vel6OD61HnxNeIyjuIZGulD2yS0bVQYPxE7a75lJZU1OePYWoT7kPmNiWOaAV51MkDxjSZgdglDJBsKGS80B2/yM8i8H+gGCPpH39SLuatl1VY51r30HWJWg1E8BRjMrkKTiRgKlCquRdKqo+kX+ZNu4GC9MoRVOx/FTgYOJJl2BYP0rEJWoTleUSEZjouywa2mkiiKKAEpXm1JzQxE7ukB45fzcdtxPYcQjGw5bJ6AxnfYwycPNCjJ2slBseV5EDivhJ1AG9I9gD4i5Rpcy3oRiSx1n0qC0vkWctSZVW5tNCPbUQ/nelOd/hBZYuFyq5nyqx8UdsYCMwtOq111ki6ft1xp1JzDylQyhYIqUnOpbZE/4gBaFSb21h68Fb/GHnL+9jswtRFym/nWmvj46O4v/FAT+va9lDDii4KIEF/WI6G8QN88QpUsXcnAKmbKcOspCg4+dYYrOecgSjZrggUq14T6IZX8CVlE8ft4fVKrOAooWrOgnQfVxZHkuua/ySE9alsYDCqGJolYByys9r2UMbwiWFMtWAx9FTkfhacpPLD2r2rI71+O4WsDF/eQK8JJ5WZCS3pHjElLautH8unNSMV1uOKw7nYjloQTA4vaw+Tc46td2B3CeFcx1YUJyHxrR7coexBShcfCacyIz81G4dwc43vf6Obm6NP78+YAdM2aeicyFKU+gZfZhkiZw3vGi7buQCTLSSFOzIUwhg6aADLw2RXIqp5nN7xkohjYt/LLyX+u1ySGJ4QSXaFIN7xY0W8KPuJdpev9Zfv4S31cjFjXcw0YmzqATz0JNRuNT1RbjvRNcXpXaidhaW+WYU9PCK3D5VxWh8DYmWUy+XwNshOGYJRfLCCPiDcxx2bwS/MSPdNn1HxnCr65DBwrcfbG1+un1tZVfDq0lbaKnrOgDz8iYT40lvqtO03eAlfPP0mu1XPYd8j5IJOrhqmnnGnS8Q1GYVs/ZKUeOj1vGn4gbP2I3207KqmKdh1cC6FmPoWnzVjoYGM+Ws1URYdxqal7Q04fnFFUsydRIYmYumwtg4PIMbEYcLs4OJS3fSYw2rTmJNQsfBAeYK125HRyQ5NFeQczY/CG/fGXIwmmXODg8FkceB40tH6MaWYZXpOJ1SWRs/nD98MD9x6HAer7HKk/yVoKuqE7gdv7Htazf7n95TrxaCSQ2FjY7oJYY7kJ8Yzx/cZTbWidEoxk3pOOJ4BO9gfOLxYS3r9vHRhidbib0E2Q5v1dqtIUUIB46463gyjD8A0EKhMLQlIPIngooth8MqQ15DGvLjeEOGPWAMhZfvUQxpVzDGJb/II0GrH+PZXn612OXTvY33th6+OZTgfJNjVG5yDB6iMD6sOfqqu2BkF36hlvxY3MCmh+S0ogvf5MEfvFKOZyMkLpaTamjoVClKlgdDhoQjpg94oAO8TiwsuEJhmWvThnYpQ3ejHdc5MRx90VSWbm9//wHelEcbCf03vkMRdHc1PLvKT2j1/owk62+eHr6k8c603Wfx6KvUMAZ//oveJwu9j270/nayf3M4f4jL03Yf8UfdqiaGwtgn3ghlkCbCMaMEWKm1SxRj4CcCRVlOrbDXzUXgtMy9IRiOjW8NkVcjEHaLaqIfNITfcvHwQcUQj85l55Uj1umzp9nDpWjIgXOZH3wuG0zzPBvHm4HiBBOx8WuK2y3S16moh7czR8/RpqO6whLEWLQZXZMhoxDd/fvjIcDDVG4s4I3RgaerdcgZj2/mEV9320vRfvwGzR56/Qk7M7zX+UHJ7lri5oYu3RBDCSFeik+SuJFJJIMwQ98eOYP9+/VBN4bIW0JEik7MlSI8NQ/TSIcxFSs3upFQw5tvkOR4HkBSPi/CakViGkzDge7hbRaWIOeoeTX8GVtN3ZgfwGcUW4ndmHHAD6FrV6KwZWz+dSW6RLMrtGmIhQgDlMbSbnh4UCZzJeOxxehdOrRijJVBu5VLA3cqlzP5hnXtP7J5KUUKCFAMHqaPzY7sfh5KOe306DNQL8sdJTW5S9mqcHzTmrWcBu4YqUkJmTP6Wu+ra7215fD8BqhBvIvmN79/6kn225d+L291cWe58MC+cdORxjDNWQAIk6xNvjUbAYlVlNdETRXn2lS7E3VHIGPyqEgRp5sCC8kVRRnj5qzjO7gJxiv6A27yQfjNPzx77NlfP3fEfOrIH579zZFjGEeTu6Gcr2CCuHUELcv7ABKi8xUChvFCehbFKg3EV6Uopn+0zSfOT0VEyhyVis+MR+rKTJ8lG3b6jOcH7nbELHVWfdD5MkJCtGLFXbJdedjZzsX3VQjQcvPxNbOaUNp0EiJ9+EHcoycXdXyVHpc/6vkGDoXiIpe7jojScEFRSSdyxnXVRAw1eyovtegjdmCFABWKWRp1Jhp1lA8lLJb0Hgjf/VAlJnfM+LXIICv4Zaysq6ezQ4dmKA8xRDLbG/qAnRIEBuN7wzZLcnvOM8psKmA+qBKTfOS9i+HZd/qnPsJ7JzcuJvZmqVuxOZtTN2d5QKWuHXv+xd8dYS/+jsmQXwmbZSOA6VALQC03qSmSaZoYOTFNASsPo4z8CzgkFy0="},
    "sst2_validation.tsv": {"sha256": "a7a3bfee7fc80566578791dfea290715934cc82f436c893361c4d0108a3554f8", "bytes": 99186,
        "zlib_base64": "..."},
    "sst2_validation.metadata.json": {"sha256": "d9c018c30511f87199684de5b848a8037e968412ec101db82677663228a4e1ef", "bytes": 566,
        "zlib_base64": "eNptUstugzAQvEfKP0RcW8L6gTF8R+/IGBssOTbCkLaK8u811JWI1Jtndmdn1trH+XS5ZL1YRFBL1lwy973mN2uLwa4qe9+r0jtthq0YwoITGSZrdsFdWBP1xrtUcZHlFf4FVnTKttKvbgmRf2xkpFF8U0rfE4QNYr6h50F3VMAeTg3R6Z6CpTnZ5IPZ2YNexVjrnqr9d1SOjiPQQTnqVhs3qHmajds3BIU7ACCko5zVRP/9wChwybYGUQnSaaUqLTmUjJUVr2rUayVwDRUqa0Kl5FhTwiSvCWFI0h4QcEHKkmqeBvb+01kv+la5fvLJfFyWKTRFMer8ZubZz1fpb0mgviY/L6pvxd6KAbMcQY7wByINkIbyK2YVovAG0AAkldfaSCNsG/w6S/Visg5D3F0LqaJNkc4iFC9HcT49fwDBxJUo"}
}

DESTINATIONS = {"sst2_instruction_eval.py": EVAL_SCRIPT, "deberta_sst2.py": DEBERTA_SCRIPT,
                "sst2_validation.tsv": DATA,
                "sst2_validation.metadata.json": DATA_DIR / "sst2_validation.metadata.json"}
for name, item in EMBEDDED_ASSETS.items():
    raw = zlib.decompress(base64.b64decode(item["zlib_base64"]))
    if len(raw) != item["bytes"] or hashlib.sha256(raw).hexdigest() != item["sha256"]:
        raise RuntimeError(f"内置文件 {name} 校验失败，Notebook 可能被改动过。")
    DESTINATIONS[name].write_bytes(raw)

with DATA.open(encoding="utf-8-sig", newline="") as handle:
    labels = [row["label"] for row in csv.DictReader(handle, delimiter="\t")]
if len(labels) != 872 or labels.count("1") != 444 or labels.count("0") != 428:
    raise RuntimeError("验证集条数或标签数量不对。")
save_json(META_DIR / "embedded_assets.json",
          {name: {k: v for k, v in item.items() if k != "zlib_base64"} for name, item in EMBEDDED_ASSETS.items()})
first = labels[:N_SUBSET]
print(f"验证集 872 条（正面 444、负面 428）；前 {N_SUBSET} 条正面 {first.count('1')}、负面 {first.count('0')}。")
for name, item in EMBEDDED_ASSETS.items():
    print(f"  {name}: {item['bytes']:,} 字节，SHA-256 {item['sha256'][:16]}…")

In [ ]:
if not shutil.which("nvidia-smi"):
    raise RuntimeError("没有发现 NVIDIA GPU。请在 Settings 里把 Accelerator 设为 GPU T4 ×2 后重新运行。")
smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
gpu_lines = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines()
gpu_lines = [line for line in gpu_lines if line.startswith("GPU ")]
if smi.returncode != 0 or not gpu_lines:
    raise RuntimeError("nvidia-smi 运行失败：\n" + smi.stdout + smi.stderr)
(META_DIR / "nvidia_smi_start.txt").write_text(smi.stdout, encoding="utf-8")
OLLAMA_GPU = "0"
DEBERTA_GPU = "1" if len(gpu_lines) >= 2 else "0"
print("\n".join(gpu_lines))
if len(gpu_lines) < 2:
    note("只有一张 GPU：Ollama 和 DeBERTa 会共用 GPU 0，速度会慢一些。")
print(f"Ollama 用 GPU {OLLAMA_GPU}，DeBERTa 用 GPU {DEBERTA_GPU}。")

In [ ]:
DEBERTA_PROCESS = None
DEBERTA_LOG = DEBERTA_DIR / "train.log"
DEBERTA_MAIN = DEBERTA_DIR / "main"

if RUN_DEBERTA:
    try:
        if DEBERTA_PIP:
            # 之前 IMDB 实验里 Kaggle 自带的 torchao 0.10.0 会和新版库冲突，先卸掉；没有装也不要紧。
            subprocess.run([sys.executable, "-m", "pip", "uninstall", "--yes", "--quiet", "torchao"],
                           capture_output=True, text=True)
            pip = subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *DEBERTA_PIP],
                                 capture_output=True, text=True, timeout=1800)
            (DEBERTA_DIR / "pip_install.log").write_text(pip.stdout + pip.stderr, encoding="utf-8")
            if pip.returncode != 0:
                note("pip 安装固定版本失败，改用 Kaggle 自带的版本继续：" + pip.stderr[-500:])
        versions = subprocess.run([sys.executable, "-c",
                                   "import importlib.metadata as m\n"
                                   "for p in ('torch','transformers','datasets','sentencepiece','protobuf'):\n"
                                   "    try: print(p, m.version(p))\n"
                                   "    except Exception as e: print(p, 'missing')"],
                                  capture_output=True, text=True)
        print(versions.stdout)
        (DEBERTA_DIR / "versions.txt").write_text(versions.stdout, encoding="utf-8")

        DEBERTA_ENV = dict(os.environ, CUDA_VISIBLE_DEVICES=DEBERTA_GPU,
                           TOKENIZERS_PARALLELISM="false", PYTHONUNBUFFERED="1")
        base_command = [sys.executable, str(DEBERTA_SCRIPT), "--val-file", str(DATA)]
        if DEBERTA_TRAIN_FILE:
            base_command += ["--train-file", str(DEBERTA_TRAIN_FILE)]
        DEBERTA_MODEL = None
        for candidate in DEBERTA_MODELS:
            slug = candidate.split("/")[-1]
            print(f"冒烟测试 {candidate}：16 条训练 10 步，在前 64 条上验证……", flush=True)
            try:
                smoke = subprocess.run(base_command + ["--model", candidate, "--smoke",
                                                       "--out-dir", str(DEBERTA_DIR / f"smoke_{slug}")],
                                       env=DEBERTA_ENV, capture_output=True, text=True, timeout=2400)
                smoke_text = smoke.stdout + smoke.stderr
                smoke_ok = smoke.returncode == 0 and "SMOKE OK" in smoke.stdout
            except subprocess.TimeoutExpired as e:
                smoke_text, smoke_ok = f"冒烟测试超过 40 分钟没有结束：{e}", False
            (DEBERTA_DIR / f"smoke_{slug}.log").write_text(smoke_text, encoding="utf-8")
            print(smoke_text[-2500:])
            if smoke_ok:
                DEBERTA_MODEL = candidate
                break
            note(f"{candidate} 冒烟测试没有通过，错误见 deberta/smoke_{slug}.log。")
        if DEBERTA_MODEL:
            if DEBERTA_MODEL != DEBERTA_MODELS[0]:
                note(f"改用 {DEBERTA_MODEL} 做对照组。")
            deadline = NOTEBOOK_START + DEBERTA_BUDGET_HOURS * 3600
            log_handle = DEBERTA_LOG.open("a", encoding="utf-8")
            DEBERTA_PROCESS = subprocess.Popen(
                base_command + ["--model", DEBERTA_MODEL, "--out-dir", str(DEBERTA_MAIN),
                                "--deadline-unix", str(deadline)] + list(DEBERTA_EXTRA_ARGS),
                env=DEBERTA_ENV, stdout=log_handle, stderr=subprocess.STDOUT, start_new_session=True)
            note(f"DeBERTa（{DEBERTA_MODEL}）冒烟测试通过，已在 GPU {DEBERTA_GPU} 后台开始训练"
                 f"（进程 {DEBERTA_PROCESS.pid}）。")
        else:
            note("DeBERTa 冒烟测试都没有通过，跳过 DeBERTa，只跑 Ollama。")
    except Exception as e:
        note(f"DeBERTa 准备阶段出错，跳过 DeBERTa：{type(e).__name__}: {e}")
else:
    note("RUN_DEBERTA = False，跳过 DeBERTa。")

In [ ]:
RUN_ENV = dict(os.environ)
RUN_ENV.update(OLLAMA_HOST="127.0.0.1:11434", OLLAMA_MODELS=str(MODEL_DIR), OLLAMA_NUM_PARALLEL="1",
               OLLAMA_CONTEXT_LENGTH="8192", OLLAMA_KEEP_ALIVE="30m", OLLAMA_MAX_LOADED_MODELS="1",
               CUDA_VISIBLE_DEVICES=OLLAMA_GPU)
LOCAL_OPENER = urllib.request.build_opener(urllib.request.ProxyHandler({}))


def ollama_json(endpoint, payload=None, timeout=10):
    body = None if payload is None else json.dumps(payload).encode("utf-8")
    request = urllib.request.Request(HOST + endpoint, data=body, headers={"Content-Type": "application/json"})
    with LOCAL_OPENER.open(request, timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8"))


def version_matches(text):
    return re.search(r"(?<![0-9.])" + re.escape(OLLAMA_VERSION) + r"(?![0-9.])", text or "") is not None


def install_ollama():
    missing_tools = [name for name in ("curl", "zstd") if not shutil.which(name)]
    privilege_prefix = []
    if os.geteuid() != 0:
        if not shutil.which("sudo"):
            raise RuntimeError("安装 Ollama 需要 root 或 sudo 权限。")
        privilege_prefix = ["sudo"]
    if missing_tools:
        if not shutil.which("apt-get"):
            raise RuntimeError("缺少工具且找不到 apt-get：" + ", ".join(missing_tools))
        apt_env = dict(RUN_ENV, DEBIAN_FRONTEND="noninteractive")
        if subprocess.run(privilege_prefix + ["apt-get", "update", "-qq"], env=apt_env).returncode != 0:
            # 第三方软件源签名过期等问题会让 update 报错，但 Ubuntu 主源通常已更新，继续尝试安装。
            print("apt-get update 有报错，继续尝试安装：", ", ".join(missing_tools))
        subprocess.run(privilege_prefix + ["apt-get", "install", "-y", "-qq"] + missing_tools,
                       env=apt_env, check=True)
        still_missing = [name for name in missing_tools if not shutil.which(name)]
        if still_missing:
            raise RuntimeError("安装后仍缺少：" + ", ".join(still_missing))
    installed = ""
    if shutil.which("ollama"):
        existing = subprocess.run(["ollama", "--version"], capture_output=True, text=True, env=RUN_ENV)
        installed = existing.stdout + existing.stderr
    if not version_matches(installed):
        installer = Path("/tmp/sst2_ollama_install.sh")
        subprocess.run(["curl", "-fSL", "--retry", "3", "https://ollama.com/install.sh", "-o", str(installer)],
                       check=True, env=RUN_ENV)
        save_json(META_DIR / "ollama_install.json",
                  {"url": "https://ollama.com/install.sh", "requested_version": OLLAMA_VERSION,
                   "sha256": hashlib.sha256(installer.read_bytes()).hexdigest(),
                   "downloaded_at_utc": datetime.now(timezone.utc).isoformat()})
        subprocess.run(privilege_prefix + ["env", "OLLAMA_VERSION=" + OLLAMA_VERSION, "sh", str(installer)],
                       env=dict(RUN_ENV, OLLAMA_VERSION=OLLAMA_VERSION), check=True)
    check = subprocess.run(["ollama", "--version"], capture_output=True, text=True, env=RUN_ENV)
    cli_version = (check.stdout + check.stderr).strip()
    if not version_matches(cli_version):
        raise RuntimeError("Ollama 客户端版本不是 " + OLLAMA_VERSION + "：" + cli_version)
    return cli_version


def start_server():
    try:
        return ollama_json("/api/version", timeout=2)  # 同一会话重跑时复用已经在运行的服务
    except (urllib.error.URLError, TimeoutError, OSError):
        pass
    log_handle = (META_DIR / "ollama_server.log").open("a", encoding="utf-8")
    process = subprocess.Popen(["ollama", "serve"], env=RUN_ENV, stdout=log_handle,
                               stderr=subprocess.STDOUT, start_new_session=True)
    deadline = time.monotonic() + 120
    while True:
        if process.poll() is not None:
            raise RuntimeError("Ollama 服务启动失败：\n" + tail(META_DIR / "ollama_server.log", 3000))
        try:
            return ollama_json("/api/version", timeout=2)
        except (urllib.error.URLError, TimeoutError, OSError):
            if time.monotonic() >= deadline:
                raise RuntimeError("等待 Ollama 服务超时，见 metadata/ollama_server.log。") from None
            time.sleep(1)


OLLAMA_READY = False
if RUN_OLLAMA:
    try:
        CLI_VERSION = install_ollama()
        SERVER_VERSION = start_server()
        if SERVER_VERSION.get("version") != OLLAMA_VERSION:
            raise RuntimeError(f"服务版本 {SERVER_VERSION} 不是 {OLLAMA_VERSION}")
        save_json(META_DIR / "ollama_server.json",
                  {"endpoint": HOST, "server_version": SERVER_VERSION, "cli_version": CLI_VERSION,
                   "models_directory": str(MODEL_DIR), "python": sys.version, "platform": platform.platform(),
                   "ollama_runtime_options": {k: RUN_ENV[k] for k in (
                       "OLLAMA_HOST", "OLLAMA_MODELS", "OLLAMA_NUM_PARALLEL", "OLLAMA_CONTEXT_LENGTH",
                       "OLLAMA_KEEP_ALIVE", "OLLAMA_MAX_LOADED_MODELS", "CUDA_VISIBLE_DEVICES")},
                   "checked_at_utc": datetime.now(timezone.utc).isoformat()})
        OLLAMA_READY = True
        print("Ollama 服务已就绪：", HOST, "；版本", SERVER_VERSION["version"])
    except Exception as e:
        note(f"Ollama 安装或启动失败，跳过全部 Ollama 实验：{type(e).__name__}: {e}")
else:
    note("RUN_OLLAMA = False，跳过 Ollama。")

In [ ]:
MODEL_INFO = {}


def pull_models():
    if OLLAMA_READY:
        print(f"模型目录剩余空间 {shutil.disk_usage(MODEL_DIR).free / 2 ** 30:.1f} GB")
        for model in OLLAMA_MODELS:
            try:
                started = time.monotonic()
                pull = subprocess.run(["ollama", "pull", model], env=RUN_ENV, capture_output=True,
                                      text=True, timeout=3600)
                if pull.returncode != 0:
                    raise RuntimeError((pull.stdout + pull.stderr)[-800:])
                entry = next((item for item in ollama_json("/api/tags").get("models", [])
                              if model in (item.get("name"), item.get("model"))), None)
                if entry is None or not entry.get("digest"):
                    raise RuntimeError("下载后在 /api/tags 里找不到这个模型的 digest")
                show = ollama_json("/api/show", {"model": model}, timeout=60)
                digest = entry["digest"]
                slug = re.sub(r"[^A-Za-z0-9._-]+", "-", model) + "_" + \
                    re.sub(r"[^A-Za-z0-9]", "", digest.removeprefix("sha256:"))[:12]
                results_dir = OLLAMA_RESULTS / slug
                results_dir.mkdir(parents=True, exist_ok=True)
                MODEL_INFO[model] = {"digest": digest, "capabilities": show.get("capabilities", []),
                                     "details": show.get("details", {}), "size": entry.get("size"),
                                     "results_dir": str(results_dir),
                                     "pull_seconds": round(time.monotonic() - started, 1)}
                save_json(META_DIR / f"model_{slug}.json",
                          {"model": model, **MODEL_INFO[model], "tag_entry": entry,
                           "show": {k: v for k, v in show.items() if k not in ("modelfile", "license", "tensors")},
                           "recorded_at_utc": datetime.now(timezone.utc).isoformat()})
                print(f"{model}: {(entry.get('size') or 0) / 1e9:.1f} GB，{show.get('details', {}).get('quantization_level')}，"
                      f"能力 {MODEL_INFO[model]['capabilities']}，下载用了 {MODEL_INFO[model]['pull_seconds']:.0f} 秒")
            except Exception as e:
                note(f"{model} 下载失败，跳过它的实验：{type(e).__name__}: {' '.join(str(e).split())[-500:]}")
        save_json(META_DIR / "models.json", MODEL_INFO)


try:
    pull_models()
except Exception as e:
    note(f"下载模型时出错：{type(e).__name__}: {e}")

In [ ]:
def stop_process(process, reason):
    """让评测脚本像按了 Ctrl+C 一样退出：它会先保存断点。"""
    if process.poll() is None:
        print(f"\n[{reason}] 发送中断信号，已完成的条目会保留……", flush=True)
        process.send_signal(signal.SIGINT)
        try:
            process.wait(timeout=60)
        except subprocess.TimeoutExpired:
            process.terminate()
            try:
                process.wait(timeout=20)
            except subprocess.TimeoutExpired:
                process.kill()


def run_evaluator(extra_args, results_dir, label, cap_seconds):
    command = [sys.executable, str(EVAL_SCRIPT), "--provider", "ollama", "--base-url", HOST,
               "--data-file", str(DATA), "--workers", "1", "--timeout", "600",
               "--out-dir", str(results_dir), *extra_args]
    log_path = Path(results_dir) / f"{label}_console.log"
    started = time.monotonic()
    stopped = {"reason": None}
    with log_path.open("a", encoding="utf-8") as log:
        log.write("\n" + datetime.now(timezone.utc).isoformat() + "  " + " ".join(command) + "\n")
        process = subprocess.Popen(command, cwd=ROOT, env=RUN_ENV, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                                   errors="replace", bufsize=1, start_new_session=True)

        def on_cap():
            stopped["reason"] = "到达本组时间上限"
            stop_process(process, stopped["reason"])

        timer = threading.Timer(max(1.0, cap_seconds), on_cap)
        timer.daemon = True
        timer.start()
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            code = process.wait()
        except KeyboardInterrupt:
            stop_process(process, "手动中断")
            raise
        finally:
            timer.cancel()
            process.stdout.close()
    return code, stopped["reason"], time.monotonic() - started


PLAN_STATUS = []
ON_GPU = {}


def run_plan():
    if OLLAMA_READY:
        for i, exp in enumerate(PLAN, 1):
            record = dict(exp, index=i)
            info = MODEL_INFO.get(exp["model"])
            remaining = OLLAMA_BUDGET_HOURS * 3600 - (time.time() - NOTEBOOK_START)
            caps = (info or {}).get("capabilities") or []
            if info is None:
                record["status"] = "skipped_model_unavailable"
            elif ON_GPU.get(exp["model"]) is False:
                record["status"] = "skipped_not_on_gpu"
            elif exp["think"] == "on" and caps and "thinking" not in caps:
                record["status"] = "skipped_no_thinking"
            elif remaining < MIN_START_MINUTES * 60:
                record["status"] = "skipped_budget"
            else:
                cap = min(exp["cap_minutes"] * 60, remaining)
                rows = "全部 872 条" if exp["n"] == 0 else f"前 {exp['n']} 条"
                print("\n" + "=" * 78 + f"\n[{i}/{len(PLAN)}] {exp['group']}  {exp['model']}  模板 {exp['prompt']}  "
                      f"思考 {exp['think']}  {rows}\n本组上限 {cap / 60:.1f} 分钟；已用 {hours_used():.2f} 小时\n"
                      + "=" * 78, flush=True)
                label = f"{exp['prompt']}_think-{exp['think']}_{'all' if exp['n'] == 0 else 'n' + str(exp['n'])}"
                try:
                    code, reason, seconds = run_evaluator(
                        ["--model", exp["model"], "--prompt", exp["prompt"], "--think", exp["think"],
                         "--n", str(exp["n"]), "--show", "2"], info["results_dir"], label, cap)
                    record.update(return_code=code, minutes=round(seconds / 60, 1),
                                  status="completed" if code == 0 else ("stopped_cap" if reason else "failed"))
                except KeyboardInterrupt:
                    raise
                except Exception as e:
                    record.update(status="failed", error=f"{type(e).__name__}: {e}")
                try:  # Ollama 报告的显存占用为 0，说明模型落到了 CPU 上
                    loaded = next((m for m in ollama_json("/api/ps").get("models", [])
                                   if exp["model"] in (m.get("name"), m.get("model"))), None)
                    if loaded is not None:
                        vram = int(loaded.get("size_vram", 0) or 0)
                        record["size_vram_gb"] = round(vram / 2 ** 30, 2)
                        ON_GPU[exp["model"]] = vram > 0
                        if vram == 0:
                            note(f"{exp['model']} 没有加载到 GPU（显存占用 0），跳过它后面的组。")
                except Exception as e:
                    record["ps_error"] = f"{type(e).__name__}: {e}"
                print(f"\n→ 第 {i} 组结束：{record['status']}，用了 {record.get('minutes', 0)} 分钟", flush=True)
            if record["status"].startswith("skipped"):
                print(f"[{i}/{len(PLAN)}] 跳过 {exp['model']} {exp['prompt']} 思考 {exp['think']}：{record['status']}",
                      flush=True)
            PLAN_STATUS.append(record)
            save_json(META_DIR / "ollama_plan_status.json", PLAN_STATUS)
        try:
            save_json(META_DIR / "ollama_ps_end.json", ollama_json("/api/ps"))
            (META_DIR / "nvidia_smi_after_ollama.txt").write_text(
                subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout, encoding="utf-8")
        except Exception as e:
            print("记录结束时的 GPU 状态失败：", e)
        note(f"Ollama 实验结束，已用 {hours_used():.2f} 小时。")
    else:
        note("Ollama 没有就绪，跳过全部指令学习实验。")


try:
    run_plan()
except KeyboardInterrupt:
    raise
except Exception as e:
    import traceback
    note(f"运行 Ollama 实验时出错，后面的组没有跑（不影响打包）：{type(e).__name__}: {e}")
    print(traceback.format_exc())

In [ ]:
def wait_for_deberta():
    if DEBERTA_PROCESS is not None:
        final_deadline = NOTEBOOK_START + FINAL_WAIT_HOURS * 3600
        last_print = 0.0
        while DEBERTA_PROCESS.poll() is None and time.time() < final_deadline:
            if time.time() - last_print > 600:
                print(f"[已用 {hours_used():.2f} 小时] DeBERTa 仍在训练，最近的日志：\n" + tail(DEBERTA_LOG, 800), flush=True)
                last_print = time.time()
            time.sleep(30)
        if DEBERTA_PROCESS.poll() is None:
            note("DeBERTa 到等待上限仍未结束，停止训练；已完成的组照常汇总和打包。")
            DEBERTA_PROCESS.terminate()
            try:
                DEBERTA_PROCESS.wait(timeout=60)
            except subprocess.TimeoutExpired:
                DEBERTA_PROCESS.kill()
                DEBERTA_PROCESS.wait()
        note(f"DeBERTa 训练进程已结束，退出码 {DEBERTA_PROCESS.returncode}；已用 {hours_used():.2f} 小时。")
        print(tail(DEBERTA_LOG, 2500))
    else:
        print("这次没有运行 DeBERTa 训练。")


try:
    wait_for_deberta()
except Exception as e:
    note(f"等待 DeBERTa 时出错（不影响打包）：{type(e).__name__}: {e}")

In [ ]:
def run_prefix(model, prompt, think, n):
    """和评测脚本里 run_name 的规则一致。"""
    raw = f"ollama_{model}_{prompt}_think-{think}_{'all' if n <= 0 else f'n{n}'}"
    return re.sub(r"[^A-Za-z0-9._-]+", "-", raw)


def read_records(path):
    records = {}
    with open(path, encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                try:
                    rec = json.loads(line)
                except json.JSONDecodeError:
                    continue  # 被强行中断时最后一行可能不完整
                records[rec["idx"]] = rec
    return records


def pct(x):
    return "—" if x is None else f"{x:.2%}"


def dash(x):
    return "—" if x is None else x


def build_summary():
    overview = []
    status_of = {(s["model"], s["prompt"], s["think"], s["n"]): s for s in PLAN_STATUS}
    for exp in PLAN:
        key = (exp["model"], exp["prompt"], exp["think"], exp["n"])
        row = {"group": exp["group"], "model": exp["model"], "prompt": exp["prompt"], "think": exp["think"],
               "n": exp["n"], "target_rows": 872 if exp["n"] == 0 else exp["n"],
               "status": status_of.get(key, {}).get("status", "not_run"),
               "minutes": status_of.get(key, {}).get("minutes")}
        info = MODEL_INFO.get(exp["model"])
        files = sorted(Path(info["results_dir"]).glob(run_prefix(*key) + "_*.jsonl")) if info else []
        recs = {}
        for path in files:  # 一般只有一个文件；生成设置不同时会有多个，这里合在一起只为了预览
            recs.update(read_records(path))
        recs = list(recs.values())
        first = [r for r in recs if r["idx"] < N_SUBSET]
        timed = [r for r in recs if r.get("error") is None]
        row.update(done_rows=len(recs),
                   accuracy=sum(r["correct"] for r in recs) / len(recs) if recs else None,
                   first_n_rows=len(first),
                   first_n_accuracy=sum(r["correct"] for r in first) / len(first) if first else None,
                   invalid=sum(r["pred"] is None for r in recs),
                   request_errors=sum(r.get("error") is not None for r in recs),
                   truncated=sum(r.get("finish") == "length" for r in recs),
                   avg_seconds=round(sum(r["seconds"] for r in timed) / len(timed), 2) if timed else None,
                   avg_output_tokens=round(sum(r.get("completion_tokens") or 0 for r in timed) / len(timed), 1) if timed else None,
                   files=[p.name for p in files])
        overview.append(row)

    with open(ROOT / "ollama_overview.csv", "w", encoding="utf-8-sig", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=[k for k in overview[0] if k != "files"] + ["files"])
        writer.writeheader()
        writer.writerows([{**r, "files": ";".join(r["files"])} for r in overview])

    lines = [f"| 组 | 模型 | 模板 | 思考 | 状态 | 完成/目标 | 准确率 | 前 {N_SUBSET} 条准确率 | 无效 | 截断 | 秒/条 | 输出 token/条 |",
             "|---|---|---|---|---|---:|---:|---:|---:|---:|---:|---:|"]
    for r in overview:
        lines.append(f"| {r['group']} | {r['model']} | {r['prompt']} | {r['think']} | {r['status']} | "
                     f"{r['done_rows']}/{r['target_rows']} | {pct(r['accuracy'])} | {pct(r['first_n_accuracy'])} | "
                     f"{r['invalid']} | {r['truncated']} | {dash(r['avg_seconds'])} | {dash(r['avg_output_tokens'])} |")
    ollama_table = "\n".join(lines)

    # 前 N_SUBSET 条准确率的透视表：行是模型，列是模板 × 思考
    combos = [(p, t) for p in ("basic", "cot", "role_cot") for t in ("off", "on")]
    pivot = ["| 模型 | " + " | ".join(f"{p} 思考{t}" for p, t in combos) + " |",
             "|---|" + "---:|" * len(combos)]
    for model in OLLAMA_MODELS:
        cells = []
        for p, t in combos:
            hit = [r for r in overview if r["model"] == model and r["prompt"] == p and r["think"] == t]
            cells.append(pct(hit[0]["first_n_accuracy"]) if hit and hit[0]["first_n_rows"] == N_SUBSET else
                         (f"{pct(hit[0]['first_n_accuracy'])}（{hit[0]['first_n_rows']} 条）" if hit and hit[0]["first_n_rows"] else "—"))
        pivot.append(f"| {model} | " + " | ".join(cells) + " |")
    pivot_table = "\n".join(pivot)

    # 已完成的组导出错例，方便看模型在哪些句子上出错
    for r in overview:
        info = MODEL_INFO.get(r["model"])
        if not info:
            continue
        for csv_path in Path(info["results_dir"]).glob(run_prefix(r["model"], r["prompt"], r["think"], r["n"]) + "_*.csv"):
            if csv_path.stem.endswith("_errors"):
                continue
            with csv_path.open(encoding="utf-8-sig", newline="") as handle:
                reader = csv.DictReader(handle)
                fieldnames, errors = reader.fieldnames, [row for row in reader if row.get("correct") == "0"]
            with csv_path.with_name(csv_path.stem + "_errors.csv").open("w", encoding="utf-8-sig", newline="") as handle:
                writer = csv.DictWriter(handle, fieldnames=fieldnames)
                writer.writeheader()
                writer.writerows(errors)

    # DeBERTa：按训练计划逐组读 summary.json；被中途停止、没写出结果的组记为 not_finished
    deberta_rows = []
    plan_path = DEBERTA_MAIN / "plan.json"
    planned = json.loads(plan_path.read_text(encoding="utf-8"))["runs"] if plan_path.exists() else []
    for spec in planned:
        path = DEBERTA_MAIN / "runs" / spec["name"] / "summary.json"
        deberta_rows.append(json.loads(path.read_text(encoding="utf-8")) if path.exists()
                            else {**spec, "status": "not_finished"})
    size_order = {"16": 0, "64": 1, "256": 2, "1024": 3, "full": 4}
    dlines = ["| 训练条数 | 完成的种子 | 准确率（均值 ± 标准差） | 各种子 |", "|---|---:|---:|---|"]
    for size in sorted({str(r["size"]) for r in deberta_rows}, key=lambda s: size_order.get(s, 99)):
        group = [r for r in deberta_rows if str(r["size"]) == size]
        done = [r for r in group if r.get("status") == "completed"]
        accs = [r["accuracy"] for r in done]
        label = f"全量 {group[0]['n_train']:,}" if size == "full" else size
        if accs:
            mean = sum(accs) / len(accs)
            sd = (sum((a - mean) ** 2 for a in accs) / (len(accs) - 1)) ** 0.5 if len(accs) > 1 else 0.0
            seeds = "、".join(f"{r['seed']}: {r['accuracy']:.2%}" for r in done)
            dlines.append(f"| {label} | {len(done)}/{len(group)} | {mean:.2%} ± {sd:.2%} | {seeds} |")
        else:
            dlines.append(f"| {label} | 0/{len(group)} | — | {'、'.join(sorted({str(r.get('status')) for r in group}))} |")
    deberta_table = "\n".join(dlines) if deberta_rows else "（没有 DeBERTa 结果）"

    summary_md = "\n\n".join([
        "# SST-2 补充实验汇总",
        f"总用时 {hours_used():.2f} 小时。",
        "## Ollama：24 组概况", ollama_table,
        f"## Ollama：前 {N_SUBSET} 条准确率（行是模型，列是模板 × 思考）", pivot_table,
        f"## {globals().get('DEBERTA_MODEL') or 'DeBERTa'}：872 条验证集准确率", deberta_table,
        "## 运行记录", "\n".join(f"- [{n['hours']:.2f} 小时] {n['message']}" for n in STATUS["notes"])])
    (ROOT / "summary.md").write_text(summary_md + "\n", encoding="utf-8")
    print(summary_md)


try:
    build_summary()
except Exception as e:
    import traceback
    note(f"汇总时出错（不影响打包）：{type(e).__name__}: {e}")
    print(traceback.format_exc())

In [ ]:
try:
    (META_DIR / "nvidia_smi_end.txt").write_text(
        subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout, encoding="utf-8")
except Exception as e:
    print("记录 nvidia-smi 失败：", e)
note(f"开始打包，已用 {hours_used():.2f} 小时。")

manifest_path = META_DIR / "export_manifest.json"
allowed = {".json", ".jsonl", ".csv", ".tsv", ".md", ".txt", ".log", ".py"}
files = sorted(p for p in ROOT.rglob("*")
               if p.is_file() and not p.is_symlink() and p.suffix.lower() in allowed and p != manifest_path
               and not any(part.startswith(".") for part in p.relative_to(ROOT).parts))
save_json(manifest_path, [{"path": p.relative_to(ROOT).as_posix(), "bytes": p.stat().st_size,
                           "sha256": hashlib.sha256(p.read_bytes()).hexdigest()} for p in files])
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in files + [manifest_path]:
        archive.write(path, arcname="sst2_remaining/" + path.relative_to(ROOT).as_posix())
print(f"结果包：{ZIP_PATH}，{len(files) + 1} 个文件，{ZIP_PATH.stat().st_size / 2 ** 20:.1f} MB")
print("在这个版本的 Output 里下载 sst2_remaining_results.zip，发给 Claude。")